**Setting up the Environment**


In [ ]:
%pip install pandas opencv-python scipy huggingface_hub mediapipe scikit-learn matplotlib


In [ ]:
import sys
import pandas as pd
import cv2
import scipy
import huggingface_hub

print(f"Active Python Executable: {sys.executable}")
print("All dependencies ready!")


In [ ]:
import os
import pandas as pd
from huggingface_hub import hf_hub_download, login

# 1. Log in to Hugging Face
HF_TOKEN = "YOUR_TOKEN"
login(token=HF_TOKEN)

REPO_ID = "wengziheng/mcd_rppg"
DATA_DIR = "YOUR_LOCATION"
db_csv_path = os.path.join(DATA_DIR, "db.csv")

# 2. Load metadata and dynamically select the first 50 unique patients
df = pd.read_csv(db_csv_path)

id_col = 'patient_id' if 'patient_id' in df.columns else 'subject_id'
path_col = 'video' if 'video' in df.columns else 'video_path'

# Automatically grab the first 50 unique patient IDs
selected_patients = df[id_col].unique()[:50]
print(f"Targeting {len(selected_patients)} patients...")

# Filter rows corresponding to these 50 patients
dev_df = df[df[id_col].isin(selected_patients)].dropna(subset=[path_col]).reset_index(drop=True)

print(f"Starting batch download for {len(dev_df)} total video assets...\n")

downloaded_count = 0
failed_count = 0

for idx, row in dev_df.iterrows():
    relative_path = row[path_col].replace("\\", "/")
    full_path = os.path.join(DATA_DIR, relative_path)

    # Check if file already exists and is fully downloaded (> 1 MB)
    if os.path.exists(full_path) and (os.path.getsize(full_path) / (1024 * 1024)) > 1.0:
        print(f"[{idx+1}/{len(dev_df)}] ⏩ [EXISTS] {relative_path}")
        downloaded_count += 1
        continue

    print(f"[{idx+1}/{len(dev_df)}] ⏬ Downloading {relative_path}...")
    try:
        hf_hub_download(
            repo_id=REPO_ID,
            filename=relative_path,
            repo_type="dataset",
            local_dir=DATA_DIR,
            token=HF_TOKEN,
            force_download=False
        )
        downloaded_count += 1
        print(f"    ✔ Successfully downloaded!")
    except Exception as e:
        failed_count += 1
        print(f"    ✖ Failed: {e}")

print("\n==========================================")
print(f"DOWNLOAD SUMMARY:")
print(f" -> Successfully On Disk: {downloaded_count} / {len(dev_df)}")
print(f" -> Failed Downloads:     {failed_count}")
print("==========================================")

In [ ]:
import os
import cv2
import gc
import pandas as pd

# Release any lingering file handles
cv2.destroyAllWindows()
gc.collect()

# EDITED: Point DATA_DIR to your mounted Google Drive
DATA_DIR = "YOUR_LOCATION"
db_csv_path = os.path.join(DATA_DIR, "db.csv")

if not os.path.exists(db_csv_path):
    raise FileNotFoundError(f"db.csv not found at {db_csv_path}")

df = pd.read_csv(db_csv_path)

id_col = 'patient_id' if 'patient_id' in df.columns else 'subject_id'
path_col = 'video' if 'video' in df.columns else 'video_path'

# EDITED: Dynamically select the first 50 unique patients
selected_patients = df[id_col].unique()[:50]

dev_df = df[df[id_col].isin(selected_patients)].dropna(subset=[path_col]).reset_index(drop=True)

print(f"Targeting {len(dev_df)} video records from db.csv for verification...\n")

accessible_count = 0
missing_count = 0
corrupt_count = 0

for idx, row in dev_df.iterrows():
    relative_path = row[path_col].replace("\\", "/")
    full_path = os.path.join(DATA_DIR, relative_path)

    # Alternative path check if saved without folder sub-structure
    if not os.path.exists(full_path):
        full_path = os.path.join(DATA_DIR, os.path.basename(relative_path))

    if not os.path.exists(full_path):
        print(f"[{idx+1}/{len(dev_df)}] ✖ [MISSING] {relative_path}")
        missing_count += 1
        continue

    # EDITED: Fixed 1024 * 1024 multiplication operator
    file_size_mb = os.path.getsize(full_path) / (1024 * 1024)
    if file_size_mb < 1.0:
        print(f"[{idx+1}/{len(dev_df)}] ✖ [POINTER / TOO SMALL] {relative_path} ({file_size_mb:.3f} MB)")
        corrupt_count += 1
        continue

    # Decode first frame to verify binary readability
    cap = cv2.VideoCapture(full_path, cv2.CAP_FFMPEG)
    ret, frame = cap.read()
    cap.release()

    if ret and frame is not None:
        accessible_count += 1
        print(f"[{idx+1}/{len(dev_df)}] ✔ [ACCESSIBLE] {os.path.basename(full_path)} ({file_size_mb:.1f} MB)")
    else:
        corrupt_count += 1
        print(f"[{idx+1}/{len(dev_df)}] ✖ [DECODE FAILED] {os.path.basename(full_path)}")

print("\n==========================================")
print(f"SUMMARY CHECK RESULTS:")
print(f" -> Fully Accessible: {accessible_count} / {len(dev_df)}")
print(f" -> Missing Files:    {missing_count}")
print(f" -> Corrupt/Pointers: {corrupt_count}")
print("==========================================")

In [ ]:
!pip uninstall -y opencv-python opencv-python-headless opencv-contrib-python
!pip install opencv-contrib-python

**Extracting** the un-filtered Red, Green, and Blue and croping the video to just the face: **bold text**

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import urllib.request

# --- CONFIGURATION ---
DATA_DIR = "YOUR_LOCATION"
DB_CSV = os.path.join(DATA_DIR, "db.csv")
OUTPUT_RPPG_CSV = os.path.join(DATA_DIR, "raw_rppg_signals.csv")
XML_PATH = os.path.join(DATA_DIR, "haarcascade_frontalface_default.xml")

# 1. Download XML directly if missing locally
if not os.path.exists(XML_PATH):
    print("Downloading Haar Cascade XML file...")
    url = "https://raw.githubusercontent.com/opencv/opencv/master/data/haarcascades/haarcascade_frontalface_default.xml"
    urllib.request.urlretrieve(url, XML_PATH)

# Load classifier from local path
face_cascade = cv2.CascadeClassifier(XML_PATH)

if face_cascade.empty():
    raise RuntimeError("Failed to load cascade classifier XML file.")

def extract_rppg_from_video(full_path):
    """Extract face ROI mean RGB values per frame using OpenCV Cascade."""
    cap = cv2.VideoCapture(full_path, cv2.CAP_FFMPEG)
    fps = cap.get(cv2.CAP_PROP_FPS)
    if not fps or fps <= 0 or np.isnan(fps):
        fps = 29.90

    red_channel, green_channel, blue_channel = [], [], []

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        # Detect face bounding boxes
        faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(100, 100))

        if len(faces) > 0:
            # Pick the largest face detected in the frame
            x, y, w, h = max(faces, key=lambda b: b[2] * b[3])
            face_roi = frame_rgb[y:y+h, x:x+w]
            means = np.mean(face_roi, axis=(0, 1))
        else:
            # Fallback to center region crop
            h, w, _ = frame_rgb.shape
            center_crop = frame_rgb[int(h*0.2):int(h*0.8), int(w*0.2):int(w*0.8)]
            means = np.mean(center_crop, axis=(0, 1))

        red_channel.append(means[0])
        green_channel.append(means[1])
        blue_channel.append(means[2])

    cap.release()
    return red_channel, green_channel, blue_channel, fps

# --- MAIN EXTRACTION LOOP ---
df = pd.read_csv(DB_CSV)
id_col = 'patient_id' if 'patient_id' in df.columns else 'subject_id'
path_col = 'video' if 'video' in df.columns else 'video_path'

# Automatically grab the first 50 unique patient IDs
target_patients = df[id_col].unique()[:50]
dev_df = df[df[id_col].isin(target_patients)].dropna(subset=[path_col]).reset_index(drop=True)

all_extracted_records = []

print(f"Extracting raw rPPG signals across {len(dev_df)} videos (50 patients)...\n")

for idx, row in dev_df.iterrows():
    relative_path = row[path_col].replace("\\", "/")
    full_path = os.path.join(DATA_DIR, relative_path)

    # Alternative path check if saved without folder sub-structure
    if not os.path.exists(full_path):
        full_path = os.path.join(DATA_DIR, os.path.basename(relative_path))

    if not os.path.exists(full_path):
        print(f"[{idx+1}/{len(dev_df)}] ✖ [SKIPPED] File not found: {relative_path}")
        continue

    filename = os.path.basename(full_path)
    patient = row[id_col]

    print(f"[{idx+1}/{len(dev_df)}] Extracting rPPG for {filename} (Patient {patient})...")

    r, g, b, fps = extract_rppg_from_video(full_path)

    all_extracted_records.append({
        'patient_id': patient,
        'video_file': filename,
        'fps': fps,
        'total_frames': len(r),
        'r_signal': r,
        'g_signal': g,
        'b_signal': b
    })

    print(f"    ✔ Extracted {len(r)} frames at {fps:.2f} FPS")

# Save extracted raw signals
rppg_df = pd.DataFrame(all_extracted_records)
rppg_df.to_csv(OUTPUT_RPPG_CSV, index=False)

print("\n==========================================")
print("rPPG EXTRACTION COMPLETE")
print("==========================================")
print(f" -> Raw signals saved to: {OUTPUT_RPPG_CSV}")

In [ ]:
from scipy.signal import lfilter

def detrend_signal(signal, lambda_param=10):
    """Removes slow baseline wander using smoothness priors."""
    signal_len = len(signal)
    # Construct identity matrix for smoothness prior detrending
    H = np.identity(signal_len)
    ones = np.ones(signal_len)
    # Second-order difference matrix
    d = [1, -2, 1]
    D = np.dot(np.diag(ones[2:], k=-2) + np.diag(-2 * ones[1:], k=-1) + np.diag(ones, k=0),
               np.eye(signal_len, signal_len - 2)).T

    # Smoothness filter equation: (I - (I + lambda^2 D^T D)^-1) * signal
    try:
        coef = np.dot(np.linalg.inv(H + (lambda_param ** 2) * np.dot(D.T, D)), signal)
        return signal - coef
    except Exception:
        return signal # Fallback if matrix inversion fails on edge cases

In [ ]:
ing the file lock or syncing.")

Extracting db.csv:

In [ ]:
import os
from huggingface_hub import hf_hub_download, login

# 1. Authenticate with Hugging Face
HF_TOKEN = "YOUR_ TOKEN"
login(token=HF_TOKEN)

REPO_ID = "wengziheng/mcd_rppg"
DATA_DIR = "YOUR_LOCATION"
os.makedirs(DATA_DIR, exist_ok=True)

print("Downloading db.csv metadata from Hugging Face...")
hf_hub_download(
    repo_id=REPO_ID,
    filename="db.csv",  # or check if it's named 'mcd_rppg.csv' depending on the exact repo layout
    repo_type="dataset",
    local_dir=DATA_DIR,
    token=HF_TOKEN,
    force_download=True
)

print("✔ Successfully downloaded db.csv to Google Drive!")

Extracting ppg_sync:

In [ ]:
import os
import pandas as pd
import numpy as np
from huggingface_hub import hf_hub_download, login

# 1. Authenticate with Hugging Face
HF_TOKEN = "YOUR_TOKEN"
login(token=HF_TOKEN)

REPO_ID = "wengziheng/mcd_rppg"

# EDITED: Point DATA_DIR to your mounted Google Drive path
DATA_DIR = "YOUR_LOCATION"
OUTPUT_DIR = os.path.join(DATA_DIR, "extracted_ppg")
os.makedirs(OUTPUT_DIR, exist_ok=True)

db_csv_path = os.path.join(DATA_DIR, "db.csv")

# 2. Load metadata and dynamically filter for the first 50 patients
df = pd.read_csv(db_csv_path)

id_col = 'patient_id' if 'patient_id' in df.columns else 'subject_id'

# Dynamically locate the PPG path column (targeting ppg_sync)
ppg_cols = [c for c in ['ppg_sync'] if c in df.columns]
ppg_col = ppg_cols[0] if ppg_cols else 'ppg'

# EDITED: Automatically grab the first 50 unique patient IDs to match your cohort
selected_patients = df[id_col].unique()[:50]
dev_df = df[df[id_col].isin(selected_patients)].dropna(subset=[ppg_col]).reset_index(drop=True)

print(f"Starting batch PPG synchronization download for {len(dev_df)} individual recordings (50 patients)...\n")

success_count = 0
failed_count = 0

for idx, row in dev_df.iterrows():
    patient_id = row[id_col]
    relative_path = str(row[ppg_col]).replace("\\", "/")
    full_path = os.path.join(DATA_DIR, relative_path)

    print(f"[{idx+1}/{len(dev_df)}] Processing Subject {patient_id} ({relative_path})...")

    # Download file from Hugging Face if not present locally
    file_ready = False
    if os.path.exists(full_path) and os.path.getsize(full_path) > 0:
        file_ready = True
    else:
        try:
            hf_hub_download(
                repo_id=REPO_ID,
                filename=relative_path,
                repo_type="dataset",
                local_dir=DATA_DIR,
                token=HF_TOKEN,
                force_download=False
            )
            file_ready = True
            # EDITED: Updated print statement to explicitly reference ppg_sync
            print("    ✔ Successfully downloaded ppg_sync file!")
        except Exception as e:
            failed_count += 1
            print(f"    ✖ Download failed: {e}")

    # 3. Read raw signal array and export to individual CSV
    if file_ready:
        try:
            if full_path.endswith('.npy'):
                signal = np.load(full_path).flatten()
            else:
                signal = pd.read_csv(full_path, header=None).values.flatten()

            # Construct clean output filename capturing subject and trial state
            state_tag = row.get('state', row.get('condition', f"trial{idx}"))
            clean_filename = f"ppg_sync_patient{patient_id}_{state_tag}.csv"
            export_path = os.path.join(OUTPUT_DIR, clean_filename)

            # Format time-series DataFrame for this specific trial
            trial_df = pd.DataFrame({
                'sample_index': np.arange(len(signal)),
                'ppg_amplitude': signal,
                'patient_id': patient_id,
                'source_file': relative_path
            })

            trial_df.to_csv(export_path, index=False)
            success_count += 1
            print(f"    ✔ Exported: {os.path.join('extracted_ppg', clean_filename)}")

        except Exception as e:
            failed_count += 1
            print(f"    ✖ Error parsing file contents: {e}")

print("\n==========================================")
print("INDIVIDUAL EXPORT SUMMARY:")
print(f" -> Files Exported to 'extracted_ppg/': {success_count} / {len(dev_df)}")
print(f" -> Failures / Errors:                {failed_count}")
print("==========================================")

Quick View of db.csv

In [ ]:
import pandas as pd
import os

DATA_DIR = "/content/drive/MyDrive/rPPG_Project/data"
db_df = pd.read_csv(os.path.join(DATA_DIR, "db.csv"))
print("Columns in db.csv:", db_df.columns.tolist())
print(db_df.head(2))

EDA on BMI:

In [ ]:
from google.colab import drive
import os
import pandas as pd

# 1. Mount Google Drive (if not already mounted)
drive.mount('/content/drive')

# 2. Check if the file exists, or list what's in the folder
DATA_DIR = "/content/drive/MyDrive/rPPG_Project/data"

if os.path.exists(DATA_DIR):
    print("Files in data directory:", os.listdir(DATA_DIR))
    db_path = os.path.join(DATA_DIR, "db.csv")
    if os.path.exists(db_path):
        db_df = pd.read_csv(db_path)
        print("Successfully loaded db.csv! Columns:", db_df.columns.tolist())
    else:
        print("db.csv not found in this folder. Double check your file name!")
else:
    print(f"Directory {DATA_DIR} not found. Please check your path.")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure db_df is loaded
if 'db_df' in locals() and not db_df.empty:
    # Look for the BMI column (handle potential casing differences like 'BMI' or 'bmi')
    bmi_col = next((col for col in db_df.columns if col.lower() == 'bmi'), None)

    if bmi_col:
        bmi_series = db_df[bmi_col].dropna()

        print("==========================================")
        print("       BMI SUMMARY STATISTICS             ")
        print("==========================================")
        print(bmi_series.describe())
        print(f"\nMissing BMI values: {db_df[bmi_col].isna().sum()}")
        print(f"Total valid BMI samples: {len(bmi_series)}")
        print("==========================================")

        # --- PLOTS ---
        plt.figure(figsize=(12, 5))

        # Histogram & KDE
        plt.subplot(1, 2, 1)
        sns.histplot(bmi_series, kde=True, color='teal', bins=15)
        plt.title('BMI Distribution')
        plt.xlabel('BMI (kg/m²)')
        plt.ylabel('Frequency')

        # Boxplot for Outliers
        plt.subplot(1, 2, 2)
        sns.boxplot(y=bmi_series, color='skyblue')
        plt.title('BMI Boxplot')
        plt.ylabel('BMI (kg/m²)')

        plt.tight_layout()
        plt.show()
    else:
        print("Could not find a 'bmi' column in db.csv. Available columns:", db_df.columns.tolist())
else:
    print("Please run the Drive mount and data loading cell first!")

Chrom filter to remove shadows, lights and environmental noise:

In [ ]:
import os
import numpy as np
import pandas as pd

# --- CONFIGURATION ---
DATA_DIR = "/content/drive/MyDrive/rPPG_Project/data"
INPUT_CSV = os.path.join(DATA_DIR, "raw_rppg_signals.csv")
OUTPUT_CSV = os.path.join(DATA_DIR, "enhanced_rppg_signals.csv")

rppg_df = pd.read_csv(INPUT_CSV)

def parse_signal(signal_input):
    """Safely convert stringified signal arrays back into numpy arrays."""
    if isinstance(signal_input, np.ndarray):
        return signal_input.astype(float)
    if isinstance(signal_input, list):
        return np.array(signal_input, dtype=float)
    if isinstance(signal_input, str):
        cleaned = signal_input.strip().replace("np.float64(", "").replace(")", "").replace("[", "").replace("]", "").replace("'", "").replace('"', "")
        if not cleaned:
            return np.array([])
        try:
            parts = [float(x.strip()) for x in cleaned.split(",") if x.strip()]
            if len(parts) > 0:
                return np.array(parts)
        except ValueError:
            pass
        try:
            parts = [float(x.strip()) for x in cleaned.split() if x.strip()]
            return np.array(parts)
        except ValueError:
            pass
    return np.array([], dtype=float)

def compute_chrom_rppg(r, g, b, fps):
    """Applies Chrominance-based rPPG noise cancellation."""
    r_sig = parse_signal(r)
    g_sig = parse_signal(g)
    b_sig = parse_signal(b)

    if len(r_sig) == 0 or len(g_sig) == 0 or len(b_sig) == 0:
        return np.array([])

    # Standard CHROM projection vectors
    xs = 3.0 * r_sig - 2.0 * g_sig
    ys = 1.5 * r_sig + g_sig - 1.5 * b_sig

    # Motion-cancelling projection
    alpha = np.std(xs) / (np.std(ys) + 1e-6)
    chrom_signal = xs - alpha * ys
    return chrom_signal

# --- PROCESS ALL ROWS INSTANTLY ---
print("Applying CHROM filter to saved raw signals...")
enhanced_records = []

for idx, row in rppg_df.iterrows():
    chrom_sig = compute_chrom_rppg(row['r_signal'], row['g_signal'], row['b_signal'], row['fps'])

    enhanced_records.append({
        'patient_id': row['patient_id'],
        'video_file': row['video_file'],
        'fps': row['fps'],
        'total_frames': row['total_frames'],
        'g_signal': row['g_signal'],       # Keep original green if needed
        'chrom_signal': chrom_sig.tolist()  # Add new enhanced CHROM signal
    })

enhanced_df = pd.DataFrame(enhanced_records)
enhanced_df.to_csv(OUTPUT_CSV, index=False)

print("\n==========================================")
print("CHROM ENHANCEMENT COMPLETE!")
print("==========================================")
print(f" -> Enhanced signals saved to: {OUTPUT_CSV}")

Prediction from random forest based and Kfolds to verify on chrom filtered rppg:

In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, cross_val_score
from sklearn.preprocessing import StandardScaler

# --- CONFIGURATION ---
DATA_DIR = "/content/drive/MyDrive/rPPG_Project/data"
ENHANCED_CSV = os.path.join(DATA_DIR, "enhanced_rppg_signals.csv")
DB_CSV = os.path.join(DATA_DIR, "db.csv")

enhanced_df = pd.read_csv(ENHANCED_CSV)
db_df = pd.read_csv(DB_CSV)

def parse_signal(signal_input):
    """Safely convert signal strings back to numpy arrays."""
    if isinstance(signal_input, np.ndarray):
        return signal_input.astype(float)
    if isinstance(signal_input, list):
        return np.array(signal_input, dtype=float)
    if isinstance(signal_input, str):
        cleaned = signal_input.strip().replace("np.float64(", "").replace(")", "").replace("[", "").replace("]", "").replace("'", "").replace('"', "")
        if not cleaned:
            return np.array([])
        try:
            parts = [float(x.strip()) for x in cleaned.split(",") if x.strip()]
            if len(parts) > 0:
                return np.array(parts)
        except ValueError:
            pass
        try:
            parts = [float(x.strip()) for x in cleaned.split() if x.strip()]
            return np.array(parts)
        except ValueError:
            pass
    return np.array([], dtype=float)

def extract_morphology_features(signal, fps):
    """Extracts advanced waveform features from the CHROM rPPG signal."""
    sig = parse_signal(signal)
    if len(sig) < int(fps * 2):
        return None

    # Statistical & shape features capturing pulse contour and energy
    features = {
        'mean': np.mean(sig),
        'std': np.std(sig),
        'skew': pd.Series(sig).skew(),
        'kurtosis': pd.Series(sig).kurtosis(),
        'ptp_amplitude': np.ptp(sig),
        'rms': np.sqrt(np.mean(sig**2))
    }
    return features

# --- BUILD FEATURE MATRIX (X) AND TARGET (y) ---
X_features = []
y_bmi = []

print("Extracting morphological features for BMI regression...")

for idx, row in enhanced_df.iterrows():
    video_file = row['video_file']
    chrom_sig = row['chrom_signal']
    fps = row['fps']

    # Match with db.csv to get the target BMI
    match_row = db_df[db_df['video'].str.contains(video_file, na=False)]
    if match_row.empty or 'bmi' not in match_row.columns:
        continue

    bmi_val = match_row.iloc[0]['bmi']
    if pd.isna(bmi_val):
        continue

    feats = extract_morphology_features(chrom_sig, fps)
    if feats is not None:
        X_features.append(list(feats.values()))
        y_bmi.append(float(bmi_val))

X = np.array(X_features)
y = np.array(y_bmi)

print(f"Dataset compiled: {X.shape[0]} valid samples ready for training.")

# --- TRAIN AND EVALUATE MODEL ---
if len(X) > 10:
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    model = RandomForestRegressor(n_estimators=100, random_state=42)

    # 5-fold cross-validation evaluating Mean Absolute Error (MAE) in kg/m²
    cv = KFold(n_splits=5, shuffle=True, random_state=42)
    mae_scores = -cross_val_score(model, X_scaled, y, cv=cv, scoring='neg_mean_absolute_error')

    print("\n==========================================")
    print("CHROM-BASED BMI PREDICTION RESULTS:")
    print("==========================================")
    print(f" -> Cross-Validation MAE: {mae_scores.mean():.2f} (+/- {mae_scores.std():.2f}) kg/m²")
    print("==========================================")
else:
    print("Not enough matching samples with valid BMI data.")

Randome Forest with LOSO:

In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import LeaveOneGroupOut, cross_validate
from sklearn.preprocessing import StandardScaler

# --- CONFIGURATION ---
DATA_DIR = "/content/drive/MyDrive/rPPG_Project/data"
ENHANCED_CSV = os.path.join(DATA_DIR, "enhanced_rppg_signals.csv")
DB_CSV = os.path.join(DATA_DIR, "db.csv")

enhanced_df = pd.read_csv(ENHANCED_CSV)
db_df = pd.read_csv(DB_CSV)

def parse_signal(signal_input):
    """Safely convert signal strings back to numpy arrays."""
    if isinstance(signal_input, np.ndarray):
        return signal_input.astype(float)
    if isinstance(signal_input, list):
        return np.array(signal_input, dtype=float)
    if isinstance(signal_input, str):
        cleaned = signal_input.strip().replace("np.float64(", "").replace(")", "").replace("[", "").replace("]", "").replace("'", "").replace('"', "")
        if not cleaned:
            return np.array([])
        try:
            parts = [float(x.strip()) for x in cleaned.split(",") if x.strip()]
            if len(parts) > 0:
                return np.array(parts)
        except ValueError:
            pass
        try:
            parts = [float(x.strip()) for x in cleaned.split() if x.strip()]
            return np.array(parts)
        except ValueError:
            pass
    return np.array([], dtype=float)

def extract_morphology_features(signal, fps):
    """Extracts advanced waveform features from the CHROM rPPG signal."""
    sig = parse_signal(signal)
    if len(sig) < int(fps * 2):
        return None

    features = {
        'mean': np.mean(sig),
        'std': np.std(sig),
        'skew': pd.Series(sig).skew(),
        'kurtosis': pd.Series(sig).kurtosis(),
        'ptp_amplitude': np.ptp(sig),
        'rms': np.sqrt(np.mean(sig**2))
    }
    return features

# --- BUILD FEATURE MATRIX (X), TARGET (y), AND SUBJECT GROUPS (groups) ---
X_features = []
y_bmi = []
subject_groups = []

print("Extracting morphological features for LOSO BMI regression...")

for idx, row in enhanced_df.iterrows():
    video_file = row['video_file']
    chrom_sig = row['chrom_signal']
    fps = row['fps']

    # Match with db.csv to get target BMI and patient identifier
    match_row = db_df[db_df['video'].str.contains(video_file, na=False)]
    if match_row.empty or 'bmi' not in match_row.columns:
        continue

    bmi_val = match_row.iloc[0]['bmi']
    if pd.isna(bmi_val):
        continue

    # Extract patient identifier (adjust column name if your subject ID column is named differently, e.g., 'subject_id' or 'patient_id')
    # If subject ID is embedded in the filename or row, pull it here:
    subject_id = match_row.iloc[0].get('subject_id', row.get('subject_id', video_file))

    feats = extract_morphology_features(chrom_sig, fps)
    if feats is not None:
        X_features.append(list(feats.values()))
        y_bmi.append(float(bmi_val))
        subject_groups.append(subject_id)

X = np.array(X_features)
y = np.array(y_bmi)
groups = np.array(subject_groups)

print(f"Dataset compiled: {X.shape[0]} valid samples across {len(np.unique(groups))} unique subjects ready for training.")

# --- TRAIN AND EVALUATE MODEL USING LOSO ---
if len(X) > 10:
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    model = RandomForestRegressor(n_estimators=100, random_state=42)

    # Leave-One-Group-Out cross-validation
    logo = LeaveOneGroupOut()

    scoring = {'mae': 'neg_mean_absolute_error', 'rmse': 'neg_root_mean_squared_error'}
    cv_results = cross_validate(model, X_scaled, y, groups=groups, cv=logo, scoring=scoring)

    mae_scores = -cv_results['test_mae']
    rmse_scores = -cv_results['test_rmse']

    print("\n==========================================")
    print("CHROM-BASED BMI PREDICTION (LOSO CV RESULTS):")
    print("==========================================")
    print(f" -> LOSO MAE:  {mae_scores.mean():.2f} (+/- {mae_scores.std():.2f}) kg/m²")
    print(f" -> LOSO RMSE: {rmse_scores.mean():.2f} (+/- {rmse_scores.std():.2f}) kg/m²")
    print("==========================================")
else:
    print("Not enough matching samples with valid BMI data.")

CCN and LOSO with 50 people :

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler

# --- CONFIGURATION ---
DATA_DIR = "/content/drive/MyDrive/rPPG_Project/data"
ENHANCED_CSV = os.path.join(DATA_DIR, "enhanced_rppg_signals.csv")
DB_CSV = os.path.join(DATA_DIR, "db.csv")

enhanced_df = pd.read_csv(ENHANCED_CSV)
db_df = pd.read_csv(DB_CSV)
FIXED_LENGTH = 900  # 30 seconds at 30 fps

def parse_signal(signal_input):
    """Safely convert signal strings back to numpy arrays and pad/truncate to fixed length."""
    if isinstance(signal_input, np.ndarray):
        sig = signal_input.astype(float)
    elif isinstance(signal_input, list):
        sig = np.array(signal_input, dtype=float)
    elif isinstance(signal_input, str):
        cleaned = signal_input.strip().replace("np.float64(", "").replace(")", "").replace("[", "").replace("]", "").replace("'", "").replace('"', "")
        if not cleaned:
            return None
        try:
            parts = [float(x.strip()) for x in cleaned.split(",") if x.strip()]
            sig = np.array(parts, dtype=float)
        except ValueError:
            try:
                parts = [float(x.strip()) for x in cleaned.split() if x.strip()]
                sig = np.array(parts, dtype=float)
            except ValueError:
                return None
    else:
        return None

    if len(sig) == 0:
        return None

    # Pad or truncate to ensure uniform tensor shape for CNN
    if len(sig) < FIXED_LENGTH:
        sig = np.pad(sig, (0, FIXED_LENGTH - len(sig)), 'edge')
    else:
        sig = sig[:FIXED_LENGTH]
    return sig

# --- BUILD TENSORS ---
X_signals = []
y_bmi = []

print("Preparing CHROM signals for 1D-CNN...")
for idx, row in enhanced_df.iterrows():
    video_file = row['video_file']
    chrom_sig = row['chrom_signal']

    match_row = db_df[db_df['video'].str.contains(video_file, na=False)]
    if match_row.empty or 'bmi' not in match_row.columns:
        continue
    bmi_val = match_row.iloc[0]['bmi']
    if pd.isna(bmi_val):
        continue

    processed_sig = parse_signal(chrom_sig)
    if processed_sig is not None:
        X_signals.append(processed_sig)
        y_bmi.append(float(bmi_val))

X = np.array(X_signals, dtype=np.float32)
y = np.array(y_bmi, dtype=np.float32)

# Normalize signals per sample
scaler = StandardScaler()
X = scaler.fit_transform(X)

# Reshape for PyTorch Conv1D: [Batch, Channels=1, Length=900]
X = np.expand_dims(X, axis=1)

print(f"Dataset ready for CNN: {X.shape[0]} samples.")

# --- 1D-CNN ARCHITECTURE ---
class RPPG_BMI_Net(nn.Module):
    def __init__(self):
        super(RPPG_BMI_Net, self).__init__()
        self.features = nn.Sequential(
            nn.Conv1d(1, 16, kernel_size=7, stride=2, padding=3),
            nn.BatchNorm1d(16),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(16, 32, kernel_size=5, stride=1, padding=2),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(32, 64, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1)
        )
        self.regressor = nn.Sequential(
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        x = self.regressor(x)
        return x.squeeze()

# --- K-FOLD CROSS VALIDATION FOR CNN ---
kf = KFold(n_splits=5, shuffle=True, random_state=42)
fold_maes = []

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on device: {device}")

for fold, (train_idx, test_idx) in enumerate(kf.split(X)):
    print(f"\n--- Training Fold {fold + 1}/5 ---")

    X_train, X_test = torch.tensor(X[train_idx]), torch.tensor(X[test_idx])
    y_train, y_test = torch.tensor(y[train_idx]), torch.tensor(y[test_idx])

    train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=16, shuffle=True)

    model = RPPG_BMI_Net().to(device)
    criterion = nn.L1Loss() # MAE loss
    optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)

    # Training loop
    model.train()
    for epoch in range(25): # 25 epochs per fold
        for batch_x, batch_y in train_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            optimizer.zero_grad()
            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()

    # Evaluation
    model.eval()
    with torch.no_grad():
        test_x, test_y = X_test.to(device), y_test.to(device)
        preds = model(test_x)
        fold_mae = criterion(preds, test_y).item()
        fold_maes.append(fold_mae)
        print(f"Fold {fold + 1} MAE: {fold_mae:.2f} kg/m²")

print("\n==========================================")
print("1D-CNN CHROM-BASED BMI PREDICTION RESULTS:")
print("==========================================")
print(f" -> CNN Mean MAE: {np.mean(fold_maes):.2f} (+/- {np.std(fold_maes):.2f}) kg/m²")
print("==========================================")

XGBoost:

In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error
import xgboost as xgb

# --- CONFIGURATION ---
DATA_DIR = "/content/drive/MyDrive/rPPG_Project/data"
ENHANCED_CSV = os.path.join(DATA_DIR, "enhanced_rppg_signals.csv")
DB_CSV = os.path.join(DATA_DIR, "db.csv")

enhanced_df = pd.read_csv(ENHANCED_CSV)
db_df = pd.read_csv(DB_CSV)
FIXED_LENGTH = 900  # 30 seconds at 30 fps

def parse_signal(signal_input):
    """Safely convert signal strings back to numpy arrays and pad/truncate to fixed length."""
    if isinstance(signal_input, np.ndarray):
        sig = signal_input.astype(float)
    elif isinstance(signal_input, list):
        sig = np.array(signal_input, dtype=float)
    elif isinstance(signal_input, str):
        cleaned = signal_input.strip().replace("np.float64(", "").replace(")", "").replace("[", "").replace("]", "").replace("'", "").replace('"', "")
        if not cleaned:
            return None
        try:
            parts = [float(x.strip()) for x in cleaned.split(",") if x.strip()]
            sig = np.array(parts, dtype=float)
        except ValueError:
            try:
                parts = [float(x.strip()) for x in cleaned.split() if x.strip()]
                sig = np.array(parts, dtype=float)
            except ValueError:
                return None
    else:
        return None

    if len(sig) == 0:
        return None

    # Pad or truncate to ensure uniform length for tabular features
    if len(sig) < FIXED_LENGTH:
        sig = np.pad(sig, (0, FIXED_LENGTH - len(sig)), 'edge')
    else:
        sig = sig[:FIXED_LENGTH]
    return sig

# --- BUILD DATASET ---
X_signals = []
y_bmi = []

print("Preparing CHROM signals for XGBoost...")
for idx, row in enhanced_df.iterrows():
    video_file = row['video_file']
    chrom_sig = row['chrom_signal']

    match_row = db_df[db_df['video'].str.contains(video_file, na=False)]
    if match_row.empty or 'bmi' not in match_row.columns:
        continue
    bmi_val = match_row.iloc[0]['bmi']
    if pd.isna(bmi_val):
        continue

    processed_sig = parse_signal(chrom_sig)
    if processed_sig is not None:
        X_signals.append(processed_sig)
        y_bmi.append(float(bmi_val))

X = np.array(X_signals, dtype=np.float32)
y = np.array(y_bmi, dtype=np.float32)

# Normalize features per sample or globally
scaler = StandardScaler()
X = scaler.fit_transform(X)

print(f"Dataset ready for XGBoost: {X.shape[0]} samples, {X.shape[1]} features per sample.")

# --- K-FOLD CROSS VALIDATION FOR XGBOOST ---
kf = KFold(n_splits=5, shuffle=True, random_state=42)
fold_maes = []

for fold, (train_idx, test_idx) in enumerate(kf.split(X)):
    print(f"\n--- Training Fold {fold + 1}/5 ---")

    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    # Initialize XGBoost Regressor
    model = xgb.XGBRegressor(
        n_estimators=100,
        learning_rate=0.05,
        max_depth=4,
        random_state=42,
        objective='reg:absoluteerror' # Optimizes directly for MAE
    )

    # Train model
    model.fit(X_train, y_train)

    # Evaluation
    preds = model.predict(X_test)
    fold_mae = mean_absolute_error(y_test, preds)
    fold_maes.append(fold_mae)

    print(f"Fold {fold + 1} MAE: {fold_mae:.2f} kg/m²")

print("\n==========================================")
print("XGBOOST CHROM-BASED BMI PREDICTION RESULTS:")
print("==========================================")
print(f" -> XGBoost Mean MAE: {np.mean(fold_maes):.2f} (+/- {np.std(fold_maes):.2f}) kg/m²")
print("==========================================")

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler

# --- CONFIGURATION ---
DATA_DIR = "/content/drive/MyDrive/rPPG_Project/data"
DB_CSV = os.path.join(DATA_DIR, "db.csv")
RAW_SIGNALS_CSV = os.path.join(DATA_DIR, "raw_rppg_signals.csv") # Update if your CHROM CSV filename is different

enhanced_df = pd.read_csv(RAW_SIGNALS_CSV)
db_df = pd.read_csv(DB_CSV)
FIXED_LENGTH = 900  # 30 seconds at 30 fps

def parse_signal(signal_input):
    """Safely convert signal strings back to numpy arrays and pad/truncate to fixed length."""
    if isinstance(signal_input, np.ndarray):
        sig = signal_input.astype(float)
    elif isinstance(signal_input, list):
        sig = np.array(signal_input, dtype=float)
    elif isinstance(signal_input, str):
        cleaned = signal_input.strip().replace("np.float64(", "").replace(")", "").replace("[", "").replace("]", "").replace("'", "").replace('"', "")
        if not cleaned:
            return None
        try:
            parts = [float(x.strip()) for x in cleaned.split(",") if x.strip()]
            sig = np.array(parts, dtype=float)
        except ValueError:
            try:
                parts = [float(x.strip()) for x in cleaned.split() if x.strip()]
                sig = np.array(parts, dtype=float)
            except ValueError:
                return None
    else:
        return None

    if len(sig) == 0:
        return None

    if len(sig) < FIXED_LENGTH:
        sig = np.pad(sig, (0, FIXED_LENGTH - len(sig)), 'edge')
    else:
        sig = sig[:FIXED_LENGTH]
    return sig

# --- BUILD DATASET ---
X_signals = []
y_bmi = []

print("Preparing CHROM signals for MLP...")
for idx, row in enhanced_df.iterrows():
    video_file = row['video_file']
    # Check for chrom_signal column, fallback to g_signal if needed
    sig_col = 'chrom_signal' if 'chrom_signal' in enhanced_df.columns else 'g_signal'
    signal_data = row[sig_col]

    match_row = db_df[db_df['video'].str.contains(video_file, na=False)]
    if match_row.empty or 'bmi' not in match_row.columns:
        continue
    bmi_val = match_row.iloc[0]['bmi']
    if pd.isna(bmi_val):
        continue

    processed_sig = parse_signal(signal_data)
    if processed_sig is not None:
        X_signals.append(processed_sig)
        y_bmi.append(float(bmi_val))

X = np.array(X_signals, dtype=np.float32)
y = np.array(y_bmi, dtype=np.float32)

# Normalize features
scaler = StandardScaler()
X = scaler.fit_transform(X)

print(f"Dataset ready for MLP: {X.shape[0]} samples, {X.shape[1]} input features.")

# --- MLP ARCHITECTURE ---
class BMI_MLP(nn.Module):
    def __init__(self, input_dim):
        super(BMI_MLP, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(64, 32),
            nn.ReLU(),

            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.network(x).squeeze()

# --- K-FOLD CROSS VALIDATION FOR MLP ---
kf = KFold(n_splits=5, shuffle=True, random_state=42)
fold_maes = []

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training MLP on device: {device}")

for fold, (train_idx, test_idx) in enumerate(kf.split(X)):
    print(f"\n--- Training Fold {fold + 1}/5 ---")

    X_train, X_test = torch.tensor(X[train_idx]), torch.tensor(X[test_idx])
    y_train, y_test = torch.tensor(y[train_idx]), torch.tensor(y[test_idx])

    train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=16, shuffle=True)

    model = BMI_MLP(input_dim=X.shape[1]).to(device)
    criterion = nn.L1Loss() # MAE loss
    optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)

    # Training loop
    model.train()
    for epoch in range(30):
        for batch_x, batch_y in train_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            optimizer.zero_grad()
            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()

    # Evaluation
    model.eval()
    with torch.no_grad():
        test_x, test_y = X_test.to(device), y_test.to(device)
        preds = model(test_x)
        fold_mae = criterion(preds, test_y).item()
        fold_maes.append(fold_mae)
        print(f"Fold {fold + 1} MAE: {fold_mae:.2f} kg/m²")

print("\n==========================================")
print("MLP CHROM-BASED BMI PREDICTION RESULTS:")
print("==========================================")
print(f" -> MLP Mean MAE: {np.mean(fold_maes):.2f} (+/- {np.std(fold_maes):.2f}) kg/m²")
print("==========================================")

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler

# --- CONFIGURATION ---
DATA_DIR = "/content/drive/MyDrive/rPPG_Project/data"
DB_CSV = os.path.join(DATA_DIR, "db.csv")
RAW_SIGNALS_CSV = os.path.join(DATA_DIR, "raw_rppg_signals.csv")

enhanced_df = pd.read_csv(RAW_SIGNALS_CSV)
db_df = pd.read_csv(DB_CSV)
FIXED_LENGTH = 900  # 30 seconds at 30 fps

def parse_signal(signal_input):
    """Safely convert signal strings back to numpy arrays and pad/truncate to fixed length."""
    if isinstance(signal_input, np.ndarray):
        sig = signal_input.astype(float)
    elif isinstance(signal_input, list):
        sig = np.array(signal_input, dtype=float)
    elif isinstance(signal_input, str):
        cleaned = signal_input.strip().replace("np.float64(", "").replace(")", "").replace("[", "").replace("]", "").replace("'", "").replace('"', "")
        if not cleaned:
            return None
        try:
            parts = [float(x.strip()) for x in cleaned.split(",") if x.strip()]
            sig = np.array(parts, dtype=float)
        except ValueError:
            try:
                parts = [float(x.strip()) for x in cleaned.split() if x.strip()]
                sig = np.array(parts, dtype=float)
            except ValueError:
                return None
    else:
        return None

    if len(sig) == 0:
        return None

    if len(sig) < FIXED_LENGTH:
        sig = np.pad(sig, (0, FIXED_LENGTH - len(sig)), 'edge')
    else:
        sig = sig[:FIXED_LENGTH]
    return sig

# --- BUILD DATASET ---
X_signals = []
y_bmi = []

print("Preparing dataset for Multi-Task CNN...")
for idx, row in enhanced_df.iterrows():
    video_file = row['video_file']
    sig_col = 'chrom_signal' if 'chrom_signal' in enhanced_df.columns else 'g_signal'
    signal_data = row[sig_col]

    match_row = db_df[db_df['video'].str.contains(video_file, na=False)]
    if match_row.empty or 'bmi' not in match_row.columns:
        continue
    bmi_val = match_row.iloc[0]['bmi']
    if pd.isna(bmi_val):
        continue

    processed_sig = parse_signal(signal_data)
    if processed_sig is not None:
        X_signals.append(processed_sig)
        y_bmi.append(float(bmi_val))

X = np.array(X_signals, dtype=np.float32)
y = np.array(y_bmi, dtype=np.float32)

# Normalize inputs
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Reshape for PyTorch Conv1D: [Batch, Channels=1, Length=900]
X_tensor_in = np.expand_dims(X_scaled, axis=1)
X_tensor_target = np.expand_dims(X, axis=1) # Target reconstruction can use unscaled/normalized signal

print(f"Dataset ready: {X.shape[0]} samples.")

# --- MULTI-TASK NETWORK ARCHITECTURE ---
class MultiTaskRPPGNet(nn.Module):
    def __init__(self):
        super(MultiTaskRPPGNet, self).__init__()

        # Shared Encoder Backbone
        self.encoder = nn.Sequential(
            nn.Conv1d(1, 16, kernel_size=7, stride=2, padding=3),
            nn.BatchNorm1d(16),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(16, 32, kernel_size=5, stride=1, padding=2),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(32, 64, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU()
        )

        # Task 1 Head: BMI Regression
        self.regressor = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 1)
        )

        # Task 2 Head: Signal Reconstruction (Decoder)
        self.decoder = nn.Sequential(
            nn.ConvTranspose1d(64, 32, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),
            nn.ConvTranspose1d(32, 16, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv1d(16, 1, kernel_size=7, stride=1, padding=3)
        )

    def forward(self, x):
        features = self.encoder(x)
        bmi_pred = self.regressor(features)

        # Ensure reconstructed output matches exact fixed length (900)
        reconstructed_sig = self.decoder(features)
        if reconstructed_sig.size(2) != x.size(2):
            reconstructed_sig = nn.functional.interpolate(reconstructed_sig, size=x.size(2), mode='linear', align_corners=False)

        return bmi_pred.squeeze(), reconstructed_sig.squeeze(1)

# --- K-FOLD CROSS VALIDATION ---
kf = KFold(n_splits=5, shuffle=True, random_state=42)
fold_maes = []

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training Multi-Task Model on device: {device}")

alpha = 0.7  # Weight for BMI regression loss
beta = 0.3   # Weight for signal reconstruction loss

for fold, (train_idx, test_idx) in enumerate(kf.split(X_tensor_in)):
    print(f"\n--- Training Fold {fold + 1}/5 ---")

    X_train_in, X_test_in = torch.tensor(X_tensor_in[train_idx]), torch.tensor(X_tensor_in[test_idx])
    X_train_tgt, X_test_tgt = torch.tensor(X_tensor_target[train_idx]), torch.tensor(X_tensor_target[test_idx])
    y_train, y_test = torch.tensor(y[train_idx]), torch.tensor(y[test_idx])

    train_loader = DataLoader(TensorDataset(X_train_in, X_train_tgt, y_train), batch_size=16, shuffle=True)

    model = MultiTaskRPPGNet().to(device)
    criterion_reg = nn.L1Loss()  # MAE for BMI
    criterion_recon = nn.MSELoss() # MSE for signal reconstruction
    optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)

    # Training loop
    model.train()
    for epoch in range(30):
        for batch_in, batch_tgt, batch_y in train_loader:
            batch_in, batch_tgt, batch_y = batch_in.to(device), batch_tgt.to(device), batch_y.to(device)

            optimizer.zero_grad()
            bmi_preds, recon_preds = model(batch_in)

            # Combined Loss Calculation
            loss_reg = criterion_reg(bmi_preds, batch_y)
            loss_recon = criterion_recon(recon_preds, batch_tgt)
            total_loss = (alpha * loss_reg) + (beta * loss_recon)

            total_loss.backward()
            optimizer.step()

    # Evaluation
    model.eval()
    with torch.no_grad():
        test_x = X_test_in.to(device)
        test_y = y_test.to(device)
        preds, _ = model(test_x)
        fold_mae = criterion_reg(preds, test_y).item()
        fold_maes.append(fold_mae)
        print(f"Fold {fold + 1} MAE: {fold_mae:.2f} kg/m²")

print("\n==========================================")
print("MULTI-TASK MODEL CHROM-BASED RESULTS:")
print("==========================================")
print(f" -> Multi-Task Mean MAE: {np.mean(fold_maes):.2f} (+/- {np.std(fold_maes):.2f}) kg/m²")
print("==========================================")

**CNN and LOSO on 7 people:**

In [ ]:
import os
import re
import ast
import pandas as pd
import numpy as np
import scipy.signal as signal
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import mean_absolute_error, mean_squared_error

# --- 1. CONFIGURATION & PATHS ---
DATA_DIR = "/content/drive/MyDrive/Colab Notebooks/rPPG_Project/data"
RPPG_CSV = os.path.join(DATA_DIR, "raw_rppg_signals.csv")
DB_CSV = os.path.join(DATA_DIR, "db.csv")

TARGET_PATIENTS = [1020, 1091, 1097, 1099, 1107]

SIGNAL_LENGTH = 900  # 30 seconds at 30 FPS
RPPG_FPS = 30
BATCH_SIZE = 4
EPOCHS = 50
LEARNING_RATE = 1e-3
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Using device: {DEVICE}")

# --- 2. PREPROCESSING HELPERS ---
def bandpass_filter(sig, fs=RPPG_FPS, lowcut=0.7, highcut=3.5, order=3):
    """3rd-order Butterworth bandpass filter."""
    b, a = signal.butter(order, [lowcut, highcut], btype='bandpass', fs=fs)
    return signal.filtfilt(b, a, sig)

def preprocess_rppg(raw_sig, target_len=SIGNAL_LENGTH):
    """Filters, normalizes, and pads/truncates raw rPPG array to uniform length."""
    filt = bandpass_filter(raw_sig)
    norm = (filt - np.mean(filt)) / (np.std(filt) + 1e-8)

    if len(norm) > target_len:
        norm = norm[:target_len]
    elif len(norm) < target_len:
        norm = np.pad(norm, (0, target_len - len(norm)), mode='constant')

    return norm.astype(np.float32)

# --- 3. DATASET & DATALOADER ---
class RPPGBMIDataset(Dataset):
    def __init__(self, rppg_df, db_df, patient_ids, target_len=SIGNAL_LENGTH):
        self.samples = []
        self.labels = []
        self.patient_ids = []

        # Merge on 'patient_id'
        merged_df = rppg_df.merge(db_df[['patient_id', 'bmi']], on='patient_id')
        merged_df = merged_df[merged_df['patient_id'].isin(patient_ids)]

        for _, row in merged_df.iterrows():
            pid = row['patient_id']
            raw_g = row['g_signal']
            bmi_val = float(row['bmi'])

            # Parse raw rPPG string representation
            if isinstance(raw_g, str):
                cleaned_str = re.sub(r'np\.float\d*\(|\)', '', raw_g)
                try:
                    sig_arr = np.array(ast.literal_eval(cleaned_str), dtype=np.float64)
                except Exception:
                    sig_arr = np.fromstring(cleaned_str.strip("[]"), sep=',', dtype=np.float64)
                    if len(sig_arr) == 0:
                        sig_arr = np.fromstring(cleaned_str.strip("[]"), sep=' ', dtype=np.float64)
            else:
                sig_arr = np.array(raw_g, dtype=np.float64)

            processed_sig = preprocess_rppg(sig_arr, target_len=target_len)

            # Shape for Conv1D: (channels, seq_len) -> (1, 900)
            self.samples.append(np.expand_dims(processed_sig, axis=0))
            self.labels.append(bmi_val)
            self.patient_ids.append(pid)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return (
            torch.tensor(self.samples[idx], dtype=torch.float32),
            torch.tensor(self.labels[idx], dtype=torch.float32)
        )

# --- 4. 1D-CNN MODEL ARCHITECTURE ---
class RPPG_BMI_Net(nn.Module):
    def __init__(self, input_len=SIGNAL_LENGTH):
        super(RPPG_BMI_Net, self).__init__()

        self.features = nn.Sequential(
            # Block 1
            nn.Conv1d(in_channels=1, out_channels=16, kernel_size=15, stride=1, padding=7),
            nn.BatchNorm1d(16),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2),  # 900 -> 450

            # Block 2
            nn.Conv1d(in_channels=16, out_channels=32, kernel_size=9, stride=1, padding=4),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2),  # 450 -> 225

            # Block 3
            nn.Conv1d(in_channels=32, out_channels=64, kernel_size=5, stride=1, padding=2),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1)        # Global Average Pooling (64 x 1)
        )

        self.regressor = nn.Sequential(
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 1)              # Scalar BMI output
        )

    def forward(self, x):
        x = self.features(x)
        x = x.squeeze(-1)                 # (batch, 64, 1) -> (batch, 64)
        out = self.regressor(x)
        return out.squeeze(-1)            # 1D output vector

# --- 5. DATA LOADING & TRAINING LOOP ---
print("Loading datasets...")
rppg_df = pd.read_csv(RPPG_CSV)
db_df = pd.read_csv(DB_CSV)

dataset = RPPGBMIDataset(rppg_df, db_df, TARGET_PATIENTS)
print(f"Loaded {len(dataset)} patient samples for training.\n")

print("=========================================================================")
print("             1D-CNN BMI PREDICTION (LOSO CROSS-VALIDATION)              ")
print("=========================================================================")

all_targets = []
all_predictions = []

for fold, test_pid in enumerate(TARGET_PATIENTS):
    train_pids = [p for p in TARGET_PATIENTS if p != test_pid]

    train_sub = RPPGBMIDataset(rppg_df, db_df, train_pids)
    test_sub = RPPGBMIDataset(rppg_df, db_df, [test_pid])

    train_loader = DataLoader(train_sub, batch_size=BATCH_SIZE, shuffle=True)
    test_loader = DataLoader(test_sub, batch_size=1, shuffle=False)

    model = RPPG_BMI_Net().to(DEVICE)
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)

    # Train
    model.train()
    for epoch in range(EPOCHS):
        for signals, targets in train_loader:
            signals, targets = signals.to(DEVICE), targets.to(DEVICE)

            optimizer.zero_grad()
            preds = model(signals)
            loss = criterion(preds, targets)
            loss.backward()
            optimizer.step()

    # Test on unseen subject
    model.eval()
    with torch.no_grad():
        for signals, targets in test_loader:
            signals = signals.to(DEVICE)
            pred_bmi = model(signals).cpu().item()
            true_bmi = targets.item()

            all_predictions.append(pred_bmi)
            all_targets.append(true_bmi)

            print(f"Fold {fold+1} [Test Patient {test_pid}] ➔ True BMI: {true_bmi:.2f} kg/m² | Predicted BMI: {pred_bmi:.2f} kg/m² | Absolute Error: {abs(true_bmi - pred_bmi):.2f} kg/m²")

# --- 6. OVERALL METRICS ---
mae = mean_absolute_error(all_targets, all_predictions)
rmse = np.sqrt(mean_squared_error(all_targets, all_predictions))

print("-------------------------------------------------------------------------")
print(f"Overall LOSO MAE:  {mae:.2f} kg/m²")
print(f"Overall LOSO RMSE: {rmse:.2f} kg/m²")
print("=========================================================================")

Arterial Stiffness on CHROM using 1D CNN on 7 patients

In [ ]:
import os
import re
import ast
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import mean_absolute_error, mean_squared_error

# --- 1. CONFIGURATION & PATHS ---
DATA_DIR = "/content/drive/MyDrive/Colab Notebooks/rPPG_Project/data"
ENHANCED_CSV = os.path.join(DATA_DIR, "enhanced_rppg_signals.csv")
DB_CSV = os.path.join(DATA_DIR, "db.csv")

# Specify your exact patient IDs here (e.g., your 5 or 7 target subjects)
TARGET_PATIENTS = [1020, 1091, 1097, 1099, 1107]

SIGNAL_LENGTH = 900  # 30 seconds at 30 FPS
BATCH_SIZE = 4
EPOCHS = 30
LEARNING_RATE = 1e-3
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Using device: {DEVICE}")

# --- 2. PREPROCESSING HELPER ---
def parse_signal(signal_input):
    """Safely convert signal strings back to numpy arrays and pad/truncate to fixed length."""
    if isinstance(signal_input, np.ndarray):
        sig = signal_input.astype(float)
    elif isinstance(signal_input, list):
        sig = np.array(signal_input, dtype=float)
    elif isinstance(signal_input, str):
        cleaned = signal_input.strip().replace("np.float64(", "").replace(")", "").replace("[", "").replace("]", "").replace("'", "").replace('"', "")
        if not cleaned:
            return None
        try:
            parts = [float(x.strip()) for x in cleaned.split(",") if x.strip()]
            sig = np.array(parts, dtype=float)
        except ValueError:
            try:
                parts = [float(x.strip()) for x in cleaned.split() if x.strip()]
                sig = np.array(parts, dtype=float)
            except ValueError:
                return None
    else:
        return None

    if len(sig) == 0:
        return None

    if len(sig) < SIGNAL_LENGTH:
        sig = np.pad(sig, (0, SIGNAL_LENGTH - len(sig)), 'edge')
    else:
        sig = sig[:SIGNAL_LENGTH]
    return sig

# --- 3. DATASET FOR PATIENT SUBSETS ---
class RPPGRigidityDataset(Dataset):
    def __init__(self, enhanced_df, db_df, patient_ids, target_len=SIGNAL_LENGTH):
        self.samples = []
        self.labels = []
        self.patient_ids = []

        # Merge on 'patient_id'
        merged_df = enhanced_df.merge(db_df[['patient_id', 'rigidity']], on='patient_id', how='inner')
        merged_df = merged_df[merged_df['patient_id'].isin(patient_ids)]

        for _, row in merged_df.iterrows():
            pid = row['patient_id']
            chrom_sig = row['chrom_signal']
            rigidity_val = float(row['rigidity'])

            processed_sig = parse_signal(chrom_sig)
            if processed_sig is not None:
                # Normalize per sample
                norm_sig = (processed_sig - np.mean(processed_sig)) / (np.std(processed_sig) + 1e-8)
                self.samples.append(np.expand_dims(norm_sig, axis=0))
                self.labels.append(rigidity_val)
                self.patient_ids.append(pid)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return (
            torch.tensor(self.samples[idx], dtype=torch.float32),
            torch.tensor(self.labels[idx], dtype=torch.float32)
        )

# --- 4. 1D-CNN MODEL ARCHITECTURE ---
class RPPG_Rigidity_Net(nn.Module):
    def __init__(self):
        super(RPPG_Rigidity_Net, self).__init__()
        self.features = nn.Sequential(
            nn.Conv1d(1, 16, kernel_size=7, stride=2, padding=3),
            nn.BatchNorm1d(16),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(16, 32, kernel_size=5, stride=1, padding=2),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(32, 64, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1)
        )
        self.regressor = nn.Sequential(
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        x = self.regressor(x)
        return x.squeeze()

# --- 5. DATA LOADING & LOSO TRAINING LOOP ---
print("Loading datasets...")
enhanced_df = pd.read_csv(ENHANCED_CSV)
db_df = pd.read_csv(DB_CSV)

print("=========================================================================")
print("        1D-CNN ARTERIAL STIFFNESS (LOSO CROSS-VALIDATION)                ")
print("=========================================================================")

all_targets = []
all_predictions = []

for fold, test_pid in enumerate(TARGET_PATIENTS):
    train_pids = [p for p in TARGET_PATIENTS if p != test_pid]

    train_sub = RPPGRigidityDataset(enhanced_df, db_df, train_pids)
    test_sub = RPPGRigidityDataset(enhanced_df, db_df, [test_pid])

    if len(train_sub) == 0 or len(test_sub) == 0:
        print(f"Skipping Fold {fold+1} [Test Patient {test_pid}]: Insufficient data.")
        continue

    train_loader = DataLoader(train_sub, batch_size=BATCH_SIZE, shuffle=True)
    test_loader = DataLoader(test_sub, batch_size=1, shuffle=False)

    model = RPPG_Rigidity_Net().to(DEVICE)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)

    # Train on N-1 subjects
    model.train()
    for epoch in range(EPOCHS):
        for signals, targets in train_loader:
            signals, targets = signals.to(DEVICE), targets.to(DEVICE)
            optimizer.zero_grad()
            preds = model(signals)
            loss = criterion(preds, targets)
            loss.backward()
            optimizer.step()

    # Test on unseen subject
    model.eval()
    with torch.no_grad():
        for signals, targets in test_loader:
            signals, targets = signals.to(DEVICE), targets.to(DEVICE)
            pred_val = model(signals).cpu().item()
            true_val = targets.cpu().item()

            all_predictions.append(pred_val)
            all_targets.append(true_val)

            print(f"Fold {fold+1} [Test Patient {test_pid}] ➔ True Rigidity: {true_val:.4f} | Predicted Rigidity: {pred_val:.4f} | Abs Error: {abs(true_val - pred_val):.4f}")

# --- 6. OVERALL METRICS ---
if len(all_targets) > 0:
    mae = mean_absolute_error(all_targets, all_predictions)
    rmse = np.sqrt(mean_squared_error(all_targets, all_predictions))

    print("-------------------------------------------------------------------------")
    print(f"Overall LOSO MAE:  {mae:.4f}")
    print(f"Overall LOSO RMSE: {rmse:.4f}")
    print("=========================================================================")
else:
    print("No predictions made. Verify that your patient IDs match those in db.csv.")

EDA On arterial Stiffness

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# --- CONFIGURATION ---
DATA_DIR = "/content/drive/MyDrive/Colab Notebooks/rPPG_Project/data"
DB_CSV = os.path.join(DATA_DIR, "db.csv")

# Load database
db_df = pd.read_csv(DB_CSV)

# Check and extract rigidity values
if 'rigidity' in db_df.columns:
    rigidity_data = db_df['rigidity'].dropna()

    # Set up the plotting canvas
    sns.set_theme(style="whitegrid")
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # 1. Box and Whiskers Plot
    sns.boxplot(y=rigidity_data, ax=axes[0], color='skyblue')
    axes[0].set_title('Box Plot of Arterial Stiffness (Rigidity)', fontsize=12, fontweight='bold')
    axes[0].set_ylabel('Rigidity Value')

    # 2. Distribution Histogram
    sns.histplot(rigidity_data, kde=True, ax=axes[1], color='teal', bins=20)
    axes[1].set_title('Distribution Histogram of Rigidity', fontsize=12, fontweight='bold')
    axes[1].set_xlabel('Rigidity Value')
    axes[1].set_ylabel('Frequency')

    plt.tight_layout()
    plt.show()

    # Print summary statistics
    print("\n==========================================")
    print("      RIGIDITY SUMMARY STATISTICS         ")
    print("==========================================")
    print(rigidity_data.describe())
    print("==========================================")
else:
    print(f"Error: 'rigidity' column not found. Available columns in db.csv: {db_df.columns.tolist()}")

Arterial stiffness on CHROM using 1D CNN on 50 patients

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler

# --- CONFIGURATION ---
DATA_DIR = "/content/drive/MyDrive/Colab Notebooks/rPPG_Project/data"
ENHANCED_CSV = os.path.join(DATA_DIR, "enhanced_rppg_signals.csv")
DB_CSV = os.path.join(DATA_DIR, "db.csv")

enhanced_df = pd.read_csv(ENHANCED_CSV)
db_df = pd.read_csv(DB_CSV)
FIXED_LENGTH = 900  # 30 seconds at 30 fps

def parse_signal(signal_input):
    """Safely convert signal strings back to numpy arrays and pad/truncate to fixed length."""
    if isinstance(signal_input, np.ndarray):
        sig = signal_input.astype(float)
    elif isinstance(signal_input, list):
        sig = np.array(signal_input, dtype=float)
    elif isinstance(signal_input, str):
        cleaned = signal_input.strip().replace("np.float64(", "").replace(")", "").replace("[", "").replace("]", "").replace("'", "").replace('"', "")
        if not cleaned:
            return None
        try:
            parts = [float(x.strip()) for x in cleaned.split(",") if x.strip()]
            sig = np.array(parts, dtype=float)
        except ValueError:
            try:
                parts = [float(x.strip()) for x in cleaned.split() if x.strip()]
                sig = np.array(parts, dtype=float)
            except ValueError:
                return None
    else:
        return None

    if len(sig) == 0:
        return None

    # Pad or truncate to ensure uniform tensor shape for CNN
    if len(sig) < FIXED_LENGTH:
        sig = np.pad(sig, (0, FIXED_LENGTH - len(sig)), 'edge')
    else:
        sig = sig[:FIXED_LENGTH]
    return sig

# --- BUILD TENSORS ---
X_signals = []
y_rigidity = []

print("Preparing CHROM signals for Arterial Stiffness (Rigidity) 1D-CNN...")
for idx, row in enhanced_df.iterrows():
    video_file = row['video_file']
    chrom_sig = row['chrom_signal']

    match_row = db_df[db_df['video'].str.contains(video_file, na=False)]
    if match_row.empty or 'rigidity' not in match_row.columns:
        continue
    rigidity_val = match_row.iloc[0]['rigidity']
    if pd.isna(rigidity_val):
        continue

    processed_sig = parse_signal(chrom_sig)
    if processed_sig is not None:
        X_signals.append(processed_sig)
        y_rigidity.append(float(rigidity_val))

X = np.array(X_signals, dtype=np.float32)
y = np.array(y_rigidity, dtype=np.float32)

# Normalize signals per sample
scaler = StandardScaler()
X = scaler.fit_transform(X)

# Reshape for PyTorch Conv1D: [Batch, Channels=1, Length=900]
X = np.expand_dims(X, axis=1)

print(f"Dataset ready for CNN: {X.shape[0]} samples.")

# --- 1D-CNN ARCHITECTURE ---
class RPPG_Rigidity_Net(nn.Module):
    def __init__(self):
        super(RPPG_Rigidity_Net, self).__init__()
        self.features = nn.Sequential(
            nn.Conv1d(1, 16, kernel_size=7, stride=2, padding=3),
            nn.BatchNorm1d(16),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(16, 32, kernel_size=5, stride=1, padding=2),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(32, 64, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1)
        )
        self.regressor = nn.Sequential(
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        x = self.regressor(x)
        return x.squeeze()

# --- K-FOLD CROSS VALIDATION FOR CNN ---
kf = KFold(n_splits=5, shuffle=True, random_state=42)
fold_maes = []

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on device: {device}")

for fold, (train_idx, test_idx) in enumerate(kf.split(X)):
    print(f"\n--- Training Fold {fold + 1}/5 ---")

    X_train, X_test = torch.tensor(X[train_idx]), torch.tensor(X[test_idx])
    y_train, y_test = torch.tensor(y[train_idx]), torch.tensor(y[test_idx])

    train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=16, shuffle=True)

    model = RPPG_Rigidity_Net().to(device)
    criterion = nn.L1Loss() # MAE loss
    optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)

    # Training loop
    model.train()
    for epoch in range(25): # 25 epochs per fold
        for batch_x, batch_y in train_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            optimizer.zero_grad()
            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()

    # Evaluation
    model.eval()
    with torch.no_grad():
        test_x, test_y = X_test.to(device), y_test.to(device)
        preds = model(test_x)
        fold_mae = criterion(preds, test_y).item()
        fold_maes.append(fold_mae)
        print(f"Fold {fold + 1} MAE: {fold_mae:.4f}")

print("\n==========================================")
print("1D-CNN CHROM-BASED ARTERIAL STIFFNESS RESULTS:")
print("==========================================")
print(f" -> CNN Mean MAE: {np.mean(fold_maes):.4f} (+/- {np.std(fold_maes):.4f})")
print("==========================================")

Arterial Stiffness on CHROM and Random Forest

In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

# --- CONFIGURATION ---
DATA_DIR = "/content/drive/MyDrive/Colab Notebooks/rPPG_Project/data"
ENHANCED_CSV = os.path.join(DATA_DIR, "enhanced_rppg_signals.csv")
DB_CSV = os.path.join(DATA_DIR, "db.csv")

enhanced_df = pd.read_csv(ENHANCED_CSV)
db_df = pd.read_csv(DB_CSV)
FIXED_LENGTH = 900  # 30 seconds at 30 fps

def parse_signal(signal_input):
    """Safely convert signal strings back to numpy arrays and pad/truncate to fixed length."""
    if isinstance(signal_input, np.ndarray):
        sig = signal_input.astype(float)
    elif isinstance(signal_input, list):
        sig = np.array(signal_input, dtype=float)
    elif isinstance(signal_input, str):
        cleaned = signal_input.strip().replace("np.float64(", "").replace(")", "").replace("[", "").replace("]", "").replace("'", "").replace('"', "")
        if not cleaned:
            return None
        try:
            parts = [float(x.strip()) for x in cleaned.split(",") if x.strip()]
            sig = np.array(parts, dtype=float)
        except ValueError:
            try:
                parts = [float(x.strip()) for x in cleaned.split() if x.strip()]
                sig = np.array(parts, dtype=float)
            except ValueError:
                return None
    else:
        return None

    if len(sig) == 0:
        return None

    # Pad or truncate to ensure uniform feature length for Random Forest
    if len(sig) < FIXED_LENGTH:
        sig = np.pad(sig, (0, FIXED_LENGTH - len(sig)), 'edge')
    else:
        sig = sig[:FIXED_LENGTH]
    return sig

# --- BUILD FEATURE MATRIX ---
X_signals = []
y_rigidity = []

print("Preparing CHROM signals for Arterial Stiffness (Rigidity) Random Forest...")
for idx, row in enhanced_df.iterrows():
    video_file = row['video_file']
    chrom_sig = row['chrom_signal']

    match_row = db_df[db_df['video'].str.contains(video_file, na=False)]
    if match_row.empty or 'rigidity' not in match_row.columns:
        continue
    rigidity_val = match_row.iloc[0]['rigidity']
    if pd.isna(rigidity_val):
        continue

    processed_sig = parse_signal(chrom_sig)
    if processed_sig is not None:
        X_signals.append(processed_sig)
        y_rigidity.append(float(rigidity_val))

X = np.array(X_signals, dtype=np.float32)
y = np.array(y_rigidity, dtype=np.float32)

# Normalize feature vectors
scaler = StandardScaler()
X = scaler.fit_transform(X)

print(f"Dataset ready for Random Forest: {X.shape[0]} samples, {X.shape[1]} features per sample.")

# --- K-FOLD CROSS VALIDATION FOR RANDOM FOREST ---
kf = KFold(n_splits=5, shuffle=True, random_state=42)
fold_maes = []
fold_rmses = []

for fold, (train_idx, test_idx) in enumerate(kf.split(X)):
    print(f"\n--- Training Fold {fold + 1}/5 ---")

    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    # Initialize Random Forest Regressor
    model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)

    # Train model
    model.fit(X_train, y_train)

    # Evaluation
    preds = model.predict(X_test)
    fold_mae = mean_absolute_error(y_test, preds)
    fold_rmse = np.sqrt(mean_squared_error(y_test, preds))

    fold_maes.append(fold_mae)
    fold_rmses.append(fold_rmse)

    print(f"Fold {fold + 1} MAE: {fold_mae:.4f} | RMSE: {fold_rmse:.4f}")

print("\n==================================================")
print("RANDOM FOREST CHROM-BASED ARTERIAL STIFFNESS RESULTS:")
print("==================================================")
print(f" -> Mean MAE:  {np.mean(fold_maes):.4f} (+/- {np.std(fold_maes):.4f})")
print(f" -> Mean RMSE: {np.mean(fold_rmses):.4f} (+/- {np.std(fold_rmses):.4f})")
print("==================================================")

Arterial Stifness with POS and 1D CNN

In [ ]:
from google.colab import drive
import os

# --- 0. MOUNT GOOGLE DRIVE ---
print("Mounting Google Drive...")
drive.mount('/content/drive')

import re
import ast
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

# --- 1. CONFIGURATION ---
# Adjusted to the correct verified path
DATA_DIR = "/content/drive/MyDrive/rPPG_Project/data"
RAW_CSV = os.path.join(DATA_DIR, "raw_rppg_signals.csv")
DB_CSV = os.path.join(DATA_DIR, "db.csv")

raw_df = pd.read_csv(RAW_CSV)
db_df = pd.read_csv(DB_CSV)
FIXED_LENGTH = 900  # 30 seconds at 30 fps

# --- 2. POS ALGORITHM & PARSING HELPERS ---
def extract_pos_signal(rgb_data, target_len=FIXED_LENGTH):
    """
    Applies Plane-Orthogonal-to-Skin (POS) algorithm on raw RGB/color traces
    to extract the rPPG pulse signal.
    """
    if isinstance(rgb_data, list):
        rgb_data = np.array(rgb_data, dtype=float)

    if rgb_data.ndim != 2 or rgb_data.shape[1] < 3:
        if rgb_data.ndim == 1:
            sig = rgb_data
        else:
            return None
    else:
        r = rgb_data[:, 0].astype(float)
        g = rgb_data[:, 1].astype(float)
        b = rgb_data[:, 2].astype(float)

        r = r / (np.mean(r) + 1e-8)
        g = g / (np.mean(g) + 1e-8)
        b = b / (np.mean(b) + 1e-8)

        S1 = g - b
        S2 = g + b - (2 * r)

        std_s1 = np.std(S1)
        std_s2 = np.std(S2)

        if std_s2 == 0:
            sig = S1
        else:
            alpha = std_s1 / std_s2
            sig = S1 + (alpha * S2)

    if len(sig) < target_len:
        sig = np.pad(sig, (0, target_len - len(sig)), 'edge')
    else:
        sig = sig[:target_len]

    return sig.astype(np.float32)

def parse_raw_rgb(signal_input):
    """Safely parse string representations of raw signals from CSV."""
    if isinstance(signal_input, np.ndarray):
        return signal_input
    if isinstance(signal_input, list):
        return np.array(signal_input, dtype=float)
    if isinstance(signal_input, str):
        cleaned = signal_input.strip().replace("np.float64(", "").replace(")", "").replace("[", "").replace("]", "").replace("'", "").replace('"', "")
        if not cleaned:
            return None
        try:
            parts = [float(x.strip()) for x in cleaned.split(",") if x.strip()]
            return np.array(parts, dtype=float)
        except ValueError:
            try:
                parts = [float(x.strip()) for x in cleaned.split() if x.strip()]
                return np.array(parts, dtype=float)
            except ValueError:
                return None
    return None

# --- 3. BUILD TENSORS USING POS FROM RAW SIGNALS ---
X_signals = []
y_rigidity = []

print("Extracting POS signals from raw data for Arterial Stiffness (Rigidity)...")
for idx, row in raw_df.iterrows():
    video_col = 'video_file' if 'video_file' in row else 'video'
    if video_col not in row:
        continue
    video_file = row[video_col]

    raw_signal_data = None
    if 'rgb' in row and pd.notna(row['rgb']):
        raw_signal_data = parse_raw_rgb(row['rgb'])
    elif 'g_signal' in row and pd.notna(row['g_signal']):
        raw_signal_data = parse_raw_rgb(row['g_signal'])

    if raw_signal_data is None:
        continue

    match_row = db_df[db_df['video'].str.contains(str(video_file), na=False)]
    if match_row.empty or 'rigidity' not in match_row.columns:
        continue
    rigidity_val = match_row.iloc[0]['rigidity']
    if pd.isna(rigidity_val):
        continue

    pos_sig = extract_pos_signal(raw_signal_data, target_len=FIXED_LENGTH)
    if pos_sig is not None:
        X_signals.append(pos_sig)
        y_rigidity.append(float(rigidity_val))

X = np.array(X_signals, dtype=np.float32)
y = np.array(y_rigidity, dtype=np.float32)

scaler = StandardScaler()
X = scaler.fit_transform(X)
X = np.expand_dims(X, axis=1)

print(f"Dataset ready for CNN (POS from Raw): {X.shape[0]} samples.")

# --- 4. 1D-CNN ARCHITECTURE ---
class RPPG_Rigidity_Net(nn.Module):
    def __init__(self):
        super(RPPG_Rigidity_Net, self).__init__()
        self.features = nn.Sequential(
            nn.Conv1d(1, 16, kernel_size=7, stride=2, padding=3),
            nn.BatchNorm1d(16),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(16, 32, kernel_size=5, stride=1, padding=2),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(32, 64, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1)
        )
        self.regressor = nn.Sequential(
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        x = self.regressor(x)
        return x.squeeze()

# --- 5. K-FOLD CROSS VALIDATION ---
kf = KFold(n_splits=5, shuffle=True, random_state=42)
fold_mses = []
fold_rmses = []
fold_maes = []

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on device: {device}")

for fold, (train_idx, test_idx) in enumerate(kf.split(X)):
    print(f"\n--- Training Fold {fold + 1}/5 ---")

    X_train, X_test = torch.tensor(X[train_idx]), torch.tensor(X[test_idx])
    y_train, y_test = torch.tensor(y[train_idx]), torch.tensor(y[test_idx])

    train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=16, shuffle=True)

    model = RPPG_Rigidity_Net().to(device)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)

    model.train()
    for epoch in range(25):
        for batch_x, batch_y in train_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            optimizer.zero_grad()
            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()

    model.eval()
    with torch.no_grad():
        test_x, test_y = X_test.to(device), y_test.to(device)
        preds = model(test_x)

        preds_np = preds.cpu().numpy()
        y_test_np = y_test.cpu().numpy()

        fold_mse = mean_squared_error(y_test_np, preds_np)
        fold_rmse = np.sqrt(fold_mse)
        fold_mae = mean_absolute_error(y_test_np, preds_np)

        fold_mses.append(fold_mse)
        fold_rmses.append(fold_rmse)
        fold_maes.append(fold_mae)

        print(f"Fold {fold + 1} MSE: {fold_mse:.4f} | RMSE: {fold_rmse:.4f} | MAE: {fold_mae:.4f}")

print("\n==================================================")
print("1D-CNN POS (FROM RAW) ARTERIAL STIFFNESS RESULTS:")
print("==================================================")
print(f" -> Mean MSE:  {np.mean(fold_mses):.4f} (+/- {np.std(fold_mses):.4f})")
print(f" -> Mean RMSE: {np.mean(fold_rmses):.4f} (+/- {np.std(fold_rmses):.4f})")
print(f" -> Mean MAE:  {np.mean(fold_maes):.4f} (+/- {np.std(fold_maes):.4f})")
print("==================================================")

In [ ]:
import os
for root, dirs, files in os.walk("/content/drive/MyDrive"):
    if "rppg_project" in root.lower():
        print("Found path:", root)

In [ ]:
import os
for root, dirs, files in os.walk("/content/drive/MyDrive"):
    if "rppg_project" in root.lower():
        print("Found path:", root)

Arterial Stiffness with POS and random forest

In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

# --- 1. CONFIGURATION ---
DATA_DIR = "/content/drive/MyDrive/Colab Notebooks/rPPG_Project/data"
RAW_CSV = os.path.join(DATA_DIR, "raw_rppg_signals.csv")
DB_CSV = os.path.join(DATA_DIR, "db.csv")

raw_df = pd.read_csv(RAW_CSV)
db_df = pd.read_csv(DB_CSV)
FIXED_LENGTH = 900  # 30 seconds at 30 fps

# --- 2. POS ALGORITHM & PARSING HELPERS ---
def extract_pos_signal(rgb_data, target_len=FIXED_LENGTH):
    """
    Applies Plane-Orthogonal-to-Skin (POS) algorithm on raw RGB/color traces
    to extract the rPPG pulse signal.
    """
    if isinstance(rgb_data, list):
        rgb_data = np.array(rgb_data, dtype=float)

    if rgb_data.ndim != 2 or rgb_data.shape[1] < 3:
        if rgb_data.ndim == 1:
            sig = rgb_data
        else:
            return None
    else:
        r = rgb_data[:, 0].astype(float)
        g = rgb_data[:, 1].astype(float)
        b = rgb_data[:, 2].astype(float)

        # Prevent division by zero
        r = r / (np.mean(r) + 1e-8)
        g = g / (np.mean(g) + 1e-8)
        b = b / (np.mean(b) + 1e-8)

        # POS projections
        S1 = g - b
        S2 = g + b - (2 * r)

        std_s1 = np.std(S1)
        std_s2 = np.std(S2)

        if std_s2 == 0:
            sig = S1
        else:
            alpha = std_s1 / std_s2
            sig = S1 + (alpha * S2)

    # Pad or truncate to fixed length
    if len(sig) < target_len:
        sig = np.pad(sig, (0, target_len - len(sig)), 'edge')
    else:
        sig = sig[:target_len]

    return sig.astype(np.float32)

def parse_raw_rgb(signal_input):
    """Safely parse string representations of raw signals from CSV."""
    if isinstance(signal_input, np.ndarray):
        return signal_input
    if isinstance(signal_input, list):
        return np.array(signal_input, dtype=float)
    if isinstance(signal_input, str):
        cleaned = signal_input.strip().replace("np.float64(", "").replace(")", "").replace("[", "").replace("]", "").replace("'", "").replace('"', "")
        if not cleaned:
            return None
        try:
            parts = [float(x.strip()) for x in cleaned.split(",") if x.strip()]
            return np.array(parts, dtype=float)
        except ValueError:
            try:
                parts = [float(x.strip()) for x in cleaned.split() if x.strip()]
                return np.array(parts, dtype=float)
            except ValueError:
                return None
    return None

# --- 3. BUILD FEATURE MATRIX USING POS FROM RAW SIGNALS ---
X_signals = []
y_rigidity = []

print("Extracting POS signals from raw data for Random Forest...")
for idx, row in raw_df.iterrows():
    video_col = 'video_file' if 'video_file' in row else 'video'
    if video_col not in row:
        continue
    video_file = row[video_col]

    raw_signal_data = None
    if 'rgb' in row and pd.notna(row['rgb']):
        raw_signal_data = parse_raw_rgb(row['rgb'])
    elif 'g_signal' in row and pd.notna(row['g_signal']):
        raw_signal_data = parse_raw_rgb(row['g_signal'])

    if raw_signal_data is None:
        continue

    # Match with db.csv for rigidity
    match_row = db_df[db_df['video'].str.contains(str(video_file), na=False)]
    if match_row.empty or 'rigidity' not in match_row.columns:
        continue
    rigidity_val = match_row.iloc[0]['rigidity']
    if pd.isna(rigidity_val):
        continue

    # Apply POS extraction
    pos_sig = extract_pos_signal(raw_signal_data, target_len=FIXED_LENGTH)
    if pos_sig is not None:
        X_signals.append(pos_sig)
        y_rigidity.append(float(rigidity_val))

X = np.array(X_signals, dtype=np.float32)
y = np.array(y_rigidity, dtype=np.float32)

# Normalize features per sample
scaler = StandardScaler()
X = scaler.fit_transform(X)

print(f"Dataset ready for Random Forest (POS): {X.shape[0]} samples, {X.shape[1]} features.")

# --- 4. K-FOLD CROSS VALIDATION ---
kf = KFold(n_splits=5, shuffle=True, random_state=42)
fold_mses = []
fold_rmses = []
fold_maes = []

for fold, (train_idx, test_idx) in enumerate(kf.split(X)):
    print(f"\n--- Training Fold {fold + 1}/5 ---")

    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    # Initialize Random Forest Regressor
    model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)

    # Train model
    model.fit(X_train, y_train)

    # Evaluation
    preds = model.predict(X_test)

    fold_mse = mean_squared_error(y_test, preds)
    fold_rmse = np.sqrt(fold_mse)
    fold_mae = mean_absolute_error(y_test, preds)

    fold_mses.append(fold_mse)
    fold_rmses.append(fold_rmse)
    fold_maes.append(fold_mae)

    print(f"Fold {fold + 1} MSE: {fold_mse:.4f} | RMSE: {fold_rmse:.4f} | MAE: {fold_mae:.4f}")

print("\n==================================================")
print("RANDOM FOREST POS-BASED ARTERIAL STIFFNESS RESULTS:")
print("==================================================")
print(f" -> Mean MSE:  {np.mean(fold_mses):.4f} (+/- {np.std(fold_mses):.4f})")
print(f" -> Mean RMSE: {np.mean(fold_rmses):.4f} (+/- {np.std(fold_rmses):.4f})")
print(f" -> Mean MAE:  {np.mean(fold_maes):.4f} (+/- {np.std(fold_maes):.4f})")
print("==================================================")

BMI+POS+ Random Forest

In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

# --- 1. CONFIGURATION ---
# FIXED: Points to the correct subdirectory under Colab Notebooks where the CSV files exist
DATA_DIR = "/content/drive/MyDrive/Colab Notebooks/rPPG_Project/data"
RAW_CSV = os.path.join(DATA_DIR, "raw_rppg_signals.csv")
DB_CSV = os.path.join(DATA_DIR, "db.csv")

raw_df = pd.read_csv(RAW_CSV)
db_df = pd.read_csv(DB_CSV)
FIXED_LENGTH = 900  # 30 seconds at 30 fps

# --- 2. POS ALGORITHM & PARSING HELPERS ---
def extract_pos_signal(rgb_data, target_len=FIXED_LENGTH):
    """
    Applies Plane-Orthogonal-to-Skin (POS) algorithm on raw RGB/color traces
    to extract the rPPG pulse signal.
    """
    if isinstance(rgb_data, list):
        rgb_data = np.array(rgb_data, dtype=float)

    if rgb_data.ndim != 2 or rgb_data.shape[1] < 3:
        if rgb_data.ndim == 1:
            sig = rgb_data
        else:
            return None
    else:
        r = rgb_data[:, 0].astype(float)
        g = rgb_data[:, 1].astype(float)
        b = rgb_data[:, 2].astype(float)

        # Prevent division by zero
        r = r / (np.mean(r) + 1e-8)
        g = g / (np.mean(g) + 1e-8)
        b = b / (np.mean(b) + 1e-8)

        # POS projections
        S1 = g - b
        S2 = g + b - (2 * r)

        std_s1 = np.std(S1)
        std_s2 = np.std(S2)

        if std_s2 == 0:
            sig = S1
        else:
            alpha = std_s1 / std_s2
            sig = S1 + (alpha * S2)

    # Pad or truncate to fixed length
    if len(sig) < target_len:
        sig = np.pad(sig, (0, target_len - len(sig)), 'edge')
    else:
        sig = sig[:target_len]

    return sig.astype(np.float32)

def parse_raw_rgb(signal_input):
    """Safely parse string representations of raw signals from CSV."""
    if isinstance(signal_input, np.ndarray):
        return signal_input
    if isinstance(signal_input, list):
        return np.array(signal_input, dtype=float)
    if isinstance(signal_input, str):
        cleaned = signal_input.strip().replace("np.float64(", "").replace(")", "").replace("[", "").replace("]", "").replace("'", "").replace('"', "")
        if not cleaned:
            return None
        try:
            parts = [float(x.strip()) for x in cleaned.split(",") if x.strip()]
            return np.array(parts, dtype=float)
        except ValueError:
            try:
                parts = [float(x.strip()) for x in cleaned.split() if x.strip()]
                return np.array(parts, dtype=float)
            except ValueError:
                return None
    return None

# --- 3. BUILD FEATURE MATRIX USING POS FROM RAW SIGNALS ---
X_signals = []
y_bmi = []

print("Extracting POS signals from raw data for Random Forest (BMI Prediction)...")
for idx, row in raw_df.iterrows():
    video_col = 'video_file' if 'video_file' in row else 'video'
    if video_col not in row:
        continue
    video_file = row[video_col]

    raw_signal_data = None
    if 'rgb' in row and pd.notna(row['rgb']):
        raw_signal_data = parse_raw_rgb(row['rgb'])
    elif 'g_signal' in row and pd.notna(row['g_signal']):
        raw_signal_data = parse_raw_rgb(row['g_signal'])

    if raw_signal_data is None:
        continue

    # Match with db.csv for BMI
    match_row = db_df[db_df['video'].str.contains(str(video_file), na=False)]
    if match_row.empty or 'bmi' not in match_row.columns:
        continue
    bmi_val = match_row.iloc[0]['bmi']
    if pd.isna(bmi_val):
        continue

    # Apply POS extraction
    pos_sig = extract_pos_signal(raw_signal_data, target_len=FIXED_LENGTH)
    if pos_sig is not None:
        X_signals.append(pos_sig)
        y_bmi.append(float(bmi_val))

X = np.array(X_signals, dtype=np.float32)
y = np.array(y_bmi, dtype=np.float32)

# Normalize features per sample
scaler = StandardScaler()
X = scaler.fit_transform(X)

print(f"Dataset ready for Random Forest (POS): {X.shape[0]} samples, {X.shape[1]} features.")

# --- 4. K-FOLD CROSS VALIDATION ---
kf = KFold(n_splits=5, shuffle=True, random_state=42)
fold_mses = []
fold_rmses = []
fold_maes = []

for fold, (train_idx, test_idx) in enumerate(kf.split(X)):
    print(f"\n--- Training Fold {fold + 1}/5 ---")

    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    # Initialize Random Forest Regressor
    model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)

    # Train model
    model.fit(X_train, y_train)

    # Evaluation
    preds = model.predict(X_test)

    fold_mse = mean_squared_error(y_test, preds)
    fold_rmse = np.sqrt(fold_mse)
    fold_mae = mean_absolute_error(y_test, preds)

    fold_mses.append(fold_mse)
    fold_rmses.append(fold_rmse)
    fold_maes.append(fold_mae)

    print(f"Fold {fold + 1} MSE: {fold_mse:.4f} | RMSE: {fold_rmse:.4f} | MAE: {fold_mae:.4f}")

print("\n==================================================")
print("RANDOM FOREST POS-BASED BMI PREDICTION RESULTS:")
print("==================================================")
print(f" -> Mean MSE:  {np.mean(fold_mses):.4f} (+/- {np.std(fold_mses):.4f})")
print(f" -> Mean RMSE: {np.mean(fold_rmses):.4f} (+/- {np.std(fold_rmses):.4f})")
print(f" -> Mean MAE:  {np.mean(fold_maes):.4f} (+/- {np.std(fold_maes):.4f})")
print("==================================================")

BMI+POS+XGboost

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

# --- 1. CONFIGURATION ---
DATA_DIR = "/content/drive/MyDrive/Colab Notebooks/rPPG_Project/data"
RAW_CSV = os.path.join(DATA_DIR, "raw_rppg_signals.csv")
DB_CSV = os.path.join(DATA_DIR, "db.csv")

raw_df = pd.read_csv(RAW_CSV)
db_df = pd.read_csv(DB_CSV)
FIXED_LENGTH = 900  # 30 seconds at 30 fps

# --- 2. POS ALGORITHM & PARSING HELPERS ---
def extract_pos_signal(rgb_data, target_len=FIXED_LENGTH):
    if isinstance(rgb_data, list):
        rgb_data = np.array(rgb_data, dtype=float)

    if rgb_data.ndim != 2 or rgb_data.shape[1] < 3:
        if rgb_data.ndim == 1:
            sig = rgb_data
        else:
            return None
    else:
        r = rgb_data[:, 0].astype(float)
        g = rgb_data[:, 1].astype(float)
        b = rgb_data[:, 2].astype(float)

        r = r / (np.mean(r) + 1e-8)
        g = g / (np.mean(g) + 1e-8)
        b = b / (np.mean(b) + 1e-8)

        S1 = g - b
        S2 = g + b - (2 * r)

        std_s1 = np.std(S1)
        std_s2 = np.std(S2)

        if std_s2 == 0:
            sig = S1
        else:
            alpha = std_s1 / std_s2
            sig = S1 + (alpha * S2)

    if len(sig) < target_len:
        sig = np.pad(sig, (0, target_len - len(sig)), 'edge')
    else:
        sig = sig[:target_len]

    return sig.astype(np.float32)

def parse_raw_rgb(signal_input):
    if isinstance(signal_input, np.ndarray):
        return signal_input
    if isinstance(signal_input, list):
        return np.array(signal_input, dtype=float)
    if isinstance(signal_input, str):
        cleaned = signal_input.strip().replace("np.float64(", "").replace(")", "").replace("[", "").replace("]", "").replace("'", "").replace('"', "")
        if not cleaned:
            return None
        try:
            parts = [float(x.strip()) for x in cleaned.split(",") if x.strip()]
            return np.array(parts, dtype=float)
        except ValueError:
            try:
                parts = [float(x.strip()) for x in cleaned.split() if x.strip()]
                return np.array(parts, dtype=float)
            except ValueError:
                return None
    return None

# --- 3. BUILD TENSORS ---
X_signals = []
y_bmi = []

print("Extracting POS signals for 1D-CNN BMI Prediction...")
for idx, row in raw_df.iterrows():
    video_col = 'video_file' if 'video_file' in row else 'video'
    if video_col not in row:
        continue
    video_file = row[video_col]

    raw_signal_data = None
    if 'rgb' in row and pd.notna(row['rgb']):
        raw_signal_data = parse_raw_rgb(row['rgb'])
    elif 'g_signal' in row and pd.notna(row['g_signal']):
        raw_signal_data = parse_raw_rgb(row['g_signal'])

    if raw_signal_data is None:
        continue

    match_row = db_df[db_df['video'].str.contains(str(video_file), na=False)]
    if match_row.empty or 'bmi' not in match_row.columns:
        continue
    bmi_val = match_row.iloc[0]['bmi']
    if pd.isna(bmi_val):
        continue

    pos_sig = extract_pos_signal(raw_signal_data, target_len=FIXED_LENGTH)
    if pos_sig is not None:
        X_signals.append(pos_sig)
        y_bmi.append(float(bmi_val))

X = np.array(X_signals, dtype=np.float32)
y = np.array(y_bmi, dtype=np.float32)

scaler = StandardScaler()
X = scaler.fit_transform(X)
X = np.expand_dims(X, axis=1)  # Shape: [Batch, Channels=1, Length=900]

print(f"Dataset ready for CNN (POS): {X.shape[0]} samples.")

# --- 4. 1D-CNN ARCHITECTURE ---
class RPPG_BMI_Net(nn.Module):
    def __init__(self):
        super(RPPG_BMI_Net, self).__init__()
        self.features = nn.Sequential(
            nn.Conv1d(1, 16, kernel_size=7, stride=2, padding=3),
            nn.BatchNorm1d(16),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(16, 32, kernel_size=5, stride=1, padding=2),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(32, 64, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1)
        )
        self.regressor = nn.Sequential(
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        x = self.regressor(x)
        return x.squeeze()

# --- 5. K-FOLD CROSS VALIDATION ---
kf = KFold(n_splits=5, shuffle=True, random_state=42)
fold_mses, fold_rmses, fold_maes = [], [], []

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on device: {device}")

for fold, (train_idx, test_idx) in enumerate(kf.split(X)):
    print(f"\n--- Training Fold {fold + 1}/5 ---")

    X_train, X_test = torch.tensor(X[train_idx]), torch.tensor(X[test_idx])
    y_train, y_test = torch.tensor(y[train_idx]), torch.tensor(y[test_idx])

    train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=16, shuffle=True)

    model = RPPG_BMI_Net().to(device)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)

    model.train()
    for epoch in range(25):
        for batch_x, batch_y in train_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            optimizer.zero_grad()
            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()

    model.eval()
    with torch.no_grad():
        test_x, test_y = X_test.to(device), y_test.to(device)
        preds = model(test_x)

        preds_np = preds.cpu().numpy()
        y_test_np = y_test.cpu().numpy()

        fold_mse = mean_squared_error(y_test_np, preds_np)
        fold_rmse = np.sqrt(fold_mse)
        fold_mae = mean_absolute_error(y_test_np, preds_np)

        fold_mses.append(fold_mse)
        fold_rmses.append(fold_rmse)
        fold_maes.append(fold_mae)

        print(f"Fold {fold + 1} MSE: {fold_mse:.4f} | RMSE: {fold_rmse:.4f} | MAE: {fold_mae:.4f}")

print("\n==================================================")
print("1D-CNN POS-BASED BMI PREDICTION RESULTS:")
print("==================================================")
print(f" -> Mean MSE:  {np.mean(fold_mses):.4f} (+/- {np.std(fold_mses):.4f})")
print(f" -> Mean RMSE: {np.mean(fold_rmses):.4f} (+/- {np.std(fold_rmses):.4f})")
print(f" -> Mean MAE:  {np.mean(fold_maes):.4f} (+/- {np.std(fold_maes):.4f})")
print("==================================================")

BMI+ XGBOOST + POS

In [ ]:
import os
import numpy as np
import pandas as pd
from xgboost import XGBRegressor
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

# --- 1. CONFIGURATION ---
DATA_DIR = "/content/drive/MyDrive/Colab Notebooks/rPPG_Project/data"
RAW_CSV = os.path.join(DATA_DIR, "raw_rppg_signals.csv")
DB_CSV = os.path.join(DATA_DIR, "db.csv")

raw_df = pd.read_csv(RAW_CSV)
db_df = pd.read_csv(DB_CSV)
FIXED_LENGTH = 900  # 30 seconds at 30 fps

# --- 2. POS ALGORITHM & PARSING HELPERS ---
def extract_pos_signal(rgb_data, target_len=FIXED_LENGTH):
    if isinstance(rgb_data, list):
        rgb_data = np.array(rgb_data, dtype=float)

    if rgb_data.ndim != 2 or rgb_data.shape[1] < 3:
        if rgb_data.ndim == 1:
            sig = rgb_data
        else:
            return None
    else:
        r = rgb_data[:, 0].astype(float)
        g = rgb_data[:, 1].astype(float)
        b = rgb_data[:, 2].astype(float)

        r = r / (np.mean(r) + 1e-8)
        g = g / (np.mean(g) + 1e-8)
        b = b / (np.mean(b) + 1e-8)

        S1 = g - b
        S2 = g + b - (2 * r)

        std_s1 = np.std(S1)
        std_s2 = np.std(S2)

        if std_s2 == 0:
            sig = S1
        else:
            alpha = std_s1 / std_s2
            sig = S1 + (alpha * S2)

    if len(sig) < target_len:
        sig = np.pad(sig, (0, target_len - len(sig)), 'edge')
    else:
        sig = sig[:target_len]

    return sig.astype(np.float32)

def parse_raw_rgb(signal_input):
    if isinstance(signal_input, np.ndarray):
        return signal_input
    if isinstance(signal_input, list):
        return np.array(signal_input, dtype=float)
    if isinstance(signal_input, str):
        cleaned = signal_input.strip().replace("np.float64(", "").replace(")", "").replace("[", "").replace("]", "").replace("'", "").replace('"', "")
        if not cleaned:
            return None
        try:
            parts = [float(x.strip()) for x in cleaned.split(",") if x.strip()]
            return np.array(parts, dtype=float)
        except ValueError:
            try:
                parts = [float(x.strip()) for x in cleaned.split() if x.strip()]
                return np.array(parts, dtype=float)
            except ValueError:
                return None
    return None

# --- 3. BUILD FEATURE MATRIX ---
X_signals = []
y_bmi = []

print("Extracting POS signals for XGBoost BMI Prediction...")
for idx, row in raw_df.iterrows():
    video_col = 'video_file' if 'video_file' in row else 'video'
    if video_col not in row:
        continue
    video_file = row[video_col]

    raw_signal_data = None
    if 'rgb' in row and pd.notna(row['rgb']):
        raw_signal_data = parse_raw_rgb(row['rgb'])
    elif 'g_signal' in row and pd.notna(row['g_signal']):
        raw_signal_data = parse_raw_rgb(row['g_signal'])

    if raw_signal_data is None:
        continue

    match_row = db_df[db_df['video'].str.contains(str(video_file), na=False)]
    if match_row.empty or 'bmi' not in match_row.columns:
        continue
    bmi_val = match_row.iloc[0]['bmi']
    if pd.isna(bmi_val):
        continue

    pos_sig = extract_pos_signal(raw_signal_data, target_len=FIXED_LENGTH)
    if pos_sig is not None:
        X_signals.append(pos_sig)
        y_bmi.append(float(bmi_val))

X = np.array(X_signals, dtype=np.float32)
y = np.array(y_bmi, dtype=np.float32)

scaler = StandardScaler()
X = scaler.fit_transform(X)

print(f"Dataset ready for XGBoost (POS): {X.shape[0]} samples, {X.shape[1]} features.")

# --- 4. K-FOLD CROSS VALIDATION ---
kf = KFold(n_splits=5, shuffle=True, random_state=42)
fold_mses, fold_rmses, fold_maes = [], [], []

for fold, (train_idx, test_idx) in enumerate(kf.split(X)):
    print(f"\n--- Training Fold {fold + 1}/5 ---")

    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    # Initialize XGBoost Regressor
    model = XGBRegressor(n_estimators=100, learning_rate=0.05, max_depth=4, random_state=42, n_jobs=-1)
    model.fit(X_train, y_train)

    preds = model.predict(X_test)

    fold_mse = mean_squared_error(y_test, preds)
    fold_rmse = np.sqrt(fold_mse)
    fold_mae = mean_absolute_error(y_test, preds)

    fold_mses.append(fold_mse)
    fold_rmses.append(fold_rmse)
    fold_maes.append(fold_mae)

    print(f"Fold {fold + 1} MSE: {fold_mse:.4f} | RMSE: {fold_rmse:.4f} | MAE: {fold_mae:.4f}")

print("\n==================================================")
print("XGBOOST POS-BASED BMI PREDICTION RESULTS:")
print("==================================================")
print(f" -> Mean MSE:  {np.mean(fold_mses):.4f} (+/- {np.std(fold_mses):.4f})")
print(f" -> Mean RMSE: {np.mean(fold_rmses):.4f} (+/- {np.std(fold_rmses):.4f})")
print(f" -> Mean MAE:  {np.mean(fold_maes):.4f} (+/- {np.std(fold_maes):.4f})")
print("==================================================")

Arterial Stiffness + Random Forest + age + Sex + CHROM

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error

# --- CONFIGURATION ---
DATA_DIR = "/content/drive/MyDrive/Colab Notebooks/rPPG_Project/data"
ENHANCED_CSV = os.path.join(DATA_DIR, "enhanced_rppg_signals.csv")
DB_CSV = os.path.join(DATA_DIR, "db.csv")

enhanced_df = pd.read_csv(ENHANCED_CSV)
db_df = pd.read_csv(DB_CSV)
FIXED_LENGTH = 900  # 30 seconds at 30 fps

# Safely encode categorical 'sex' column if present
if 'sex' in db_df.columns:
    db_df['sex_encoded'] = LabelEncoder().fit_transform(db_df['sex'].astype(str))
else:
    db_df['sex_encoded'] = 0

def parse_signal(signal_input):
    """Safely convert signal strings back to numpy arrays and pad/truncate to fixed length."""
    if isinstance(signal_input, np.ndarray):
        sig = signal_input.astype(float)
    elif isinstance(signal_input, list):
        sig = np.array(signal_input, dtype=float)
    elif isinstance(signal_input, str):
        cleaned = signal_input.strip().replace("np.float64(", "").replace(")", "").replace("[", "").replace("]", "").replace("'", "").replace('"', "")
        if not cleaned:
            return None
        try:
            parts = [float(x.strip()) for x in cleaned.split(",") if x.strip()]
            sig = np.array(parts, dtype=float)
        except ValueError:
            try:
                parts = [float(x.strip()) for x in cleaned.split() if x.strip()]
                sig = np.array(parts, dtype=float)
            except ValueError:
                return None
    else:
        return None

    if len(sig) == 0:
        return None

    if len(sig) < FIXED_LENGTH:
        sig = np.pad(sig, (0, FIXED_LENGTH - len(sig)), 'edge')
    else:
        sig = sig[:FIXED_LENGTH]
    return sig

# --- BUILD TENSORS & DEMOGRAPHICS ---
X_signals = []
demographics_list = []
y_rigidity = []

print("Preparing CHROM signals and demographics for Arterial Stiffness (Rigidity) 1D-CNN...")
for idx, row in enhanced_df.iterrows():
    video_file = row['video_file']
    chrom_sig = row['chrom_signal']

    match_row = db_df[db_df['video'].str.contains(video_file, na=False)]
    if match_row.empty or 'rigidity' not in match_row.columns:
        continue

    rigidity_val = match_row.iloc[0]['rigidity']
    if pd.isna(rigidity_val):
        continue

    # Extract age and sex (default to 0 if missing)
    age_val = match_row.iloc[0]['age'] if 'age' in match_row.columns and pd.notna(match_row.iloc[0]['age']) else 0.0
    sex_val = match_row.iloc[0]['sex_encoded'] if 'sex_encoded' in match_row.columns else 0.0

    processed_sig = parse_signal(chrom_sig)
    if processed_sig is not None:
        X_signals.append(processed_sig)
        demographics_list.append([float(age_val), float(sex_val)])
        y_rigidity.append(float(rigidity_val))

X_sig_arr = np.array(X_signals, dtype=np.float32)
X_demo_arr = np.array(demographics_list, dtype=np.float32)
y = np.array(y_rigidity, dtype=np.float32)

# Normalize signals and demographics
scaler_sig = StandardScaler()
X_sig_scaled = scaler_sig.fit_transform(X_sig_arr)

scaler_demo = StandardScaler()
X_demo_scaled = scaler_demo.fit_transform(X_demo_arr)

# Reshape signals for PyTorch Conv1D: [Batch, Channels=1, Length=900]
X_sig_scaled = np.expand_dims(X_sig_scaled, axis=1)

print(f"Dataset ready for CNN: {X_sig_scaled.shape[0]} samples.")

# --- HYBRID 1D-CNN ARCHITECTURE (WAVEFORM + DEMOGRAPHICS) ---
class Hybrid_RPPG_Rigidity_Net(nn.Module):
    def __init__(self):
        super(Hybrid_RPPG_Rigidity_Net, self).__init__()
        self.features = nn.Sequential(
            nn.Conv1d(1, 16, kernel_size=7, stride=2, padding=3),
            nn.BatchNorm1d(16),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(16, 32, kernel_size=5, stride=1, padding=2),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(32, 64, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1)  # Output shape: [Batch, 64, 1]
        )

            # Fully connected layers take CNN output (64) + demographics (2) = 66 features
        self.regressor = nn.Sequential(
            nn.Linear(64 + 2, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 1)
        )

    def forward(self, x_sig, x_demo):
        x_feat = self.features(x_sig)
        x_feat = x_feat.view(x_feat.size(0), -1)  # Flatten CNN features

        # Concatenate waveform features with age and sex metadata
        combined = torch.cat((x_feat, x_demo), dim=1)

        out = self.regressor(combined)
        return out.squeeze()

# --- K-FOLD CROSS VALIDATION FOR HYBRID CNN ---
kf = KFold(n_splits=5, shuffle=True, random_state=42)
fold_maes = []
fold_rmses = []

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on device: {device}")

for fold, (train_idx, test_idx) in enumerate(kf.split(X_sig_scaled)):
    print(f"\n--- Training Fold {fold + 1}/5 ---")

    X_train_sig = torch.tensor(X_sig_scaled[train_idx], dtype=torch.float32)
    X_train_demo = torch.tensor(X_demo_scaled[train_idx], dtype=torch.float32)
    y_train = torch.tensor(y[train_idx], dtype=torch.float32)

    X_test_sig = torch.tensor(X_sig_scaled[test_idx], dtype=torch.float32)
    X_test_demo = torch.tensor(X_demo_scaled[test_idx], dtype=torch.float32)
    y_test = torch.tensor(y[test_idx], dtype=torch.float32)

    train_loader = DataLoader(TensorDataset(X_train_sig, X_train_demo, y_train), batch_size=16, shuffle=True)

    model = Hybrid_RPPG_Rigidity_Net().to(device)
    criterion = nn.L1Loss()  # MAE loss
    optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)

    # Training loop
    model.train()
    for epoch in range(25):
        for batch_sig, batch_demo, batch_y in train_loader:
            batch_sig, batch_demo, batch_y = batch_sig.to(device), batch_demo.to(device), batch_y.to(device)
            optimizer.zero_grad()
            outputs = model(batch_sig, batch_demo)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()

    # Evaluation
    model.eval()
    with torch.no_grad():
        test_sig_dev = X_test_sig.to(device)
        test_demo_dev = X_test_demo.to(device)
        test_y_dev = y_test.to(device)

        preds = model(test_sig_dev, test_demo_dev)

        fold_mae = mean_absolute_error(test_y_dev.cpu().numpy(), preds.cpu().numpy())
        fold_rmse = np.sqrt(mean_squared_error(test_y_dev.cpu().numpy(), preds.cpu().numpy()))

        fold_maes.append(fold_mae)
        fold_rmses.append(fold_rmse)
        print(f"Fold {fold + 1} MAE: {fold_mae:.4f} | RMSE: {fold_rmse:.4f}")

print("\n==================================================")
print("HYBRID 1D-CNN (CHROM + AGE + SEX) RESULTS:")
print("==================================================")
print(f" -> Mean MAE:  {np.mean(fold_maes):.4f} (+/- {np.std(fold_maes):.4f})")
print(f" -> Mean RMSE: {np.mean(fold_rmses):.4f} (+/- {np.std(fold_rmses):.4f})")
print("==================================================")

Arterial Stiffness + Random Forest + age + Sex + POS

In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error

# --- 1. CONFIGURATION ---
DATA_DIR = "/content/drive/MyDrive/Colab Notebooks/rPPG_Project/data"
RAW_CSV = os.path.join(DATA_DIR, "raw_rppg_signals.csv")
DB_CSV = os.path.join(DATA_DIR, "db.csv")

raw_df = pd.read_csv(RAW_CSV)
db_df = pd.read_csv(DB_CSV)
FIXED_LENGTH = 900  # 30 seconds at 30 fps

# --- 2. POS ALGORITHM & PARSING HELPERS ---
def extract_pos_signal(rgb_data, target_len=FIXED_LENGTH):
    if isinstance(rgb_data, list):
        rgb_data = np.array(rgb_data, dtype=float)

    if rgb_data.ndim != 2 or rgb_data.shape[1] < 3:
        if rgb_data.ndim == 1:
            sig = rgb_data
        else:
            return None
    else:
        r = rgb_data[:, 0].astype(float)
        g = rgb_data[:, 1].astype(float)
        b = rgb_data[:, 2].astype(float)

        r = r / (np.mean(r) + 1e-8)
        g = g / (np.mean(g) + 1e-8)
        b = b / (np.mean(b) + 1e-8)

        S1 = g - b
        S2 = g + b - (2 * r)

        std_s1 = np.std(S1)
        std_s2 = np.std(S2)

        if std_s2 == 0:
            sig = S1
        else:
            alpha = std_s1 / std_s2
            sig = S1 + (alpha * S2)

    if len(sig) < target_len:
        sig = np.pad(sig, (0, target_len - len(sig)), 'edge')
    else:
        sig = sig[:target_len]

    return sig.astype(np.float32)

def parse_raw_rgb(signal_input):
    if isinstance(signal_input, np.ndarray):
        return signal_input
    if isinstance(signal_input, list):
        return np.array(signal_input, dtype=float)
    if isinstance(signal_input, str):
        cleaned = signal_input.strip().replace("np.float64(", "").replace(")", "").replace("[", "").replace("]", "").replace("'", "").replace('"', "")
        if not cleaned:
            return None
        try:
            parts = [float(x.strip()) for x in cleaned.split(",") if x.strip()]
            return np.array(parts, dtype=float)
        except ValueError:
            try:
                parts = [float(x.strip()) for x in cleaned.split() if x.strip()]
                return np.array(parts, dtype=float)
            except ValueError:
                return None
    return None

# --- 3. BUILD MULTI-FEATURE MATRIX (POS + Age + Sex) ---
X_signals = []
demographics_list = []
y_rigidity = []

# If 'sex' is categorical, encode it safely
if 'sex' in db_df.columns:
    db_df['sex_encoded'] = LabelEncoder().fit_transform(db_df['sex'].astype(str))
else:
    db_df['sex_encoded'] = 0  # Fallback if column name differs

print("Extracting POS signals and demographic features...")
for idx, row in raw_df.iterrows():
    video_col = 'video_file' if 'video_file' in row else 'video'
    if video_col not in row:
        continue
    video_file = row[video_col]

    raw_signal_data = None
    if 'rgb' in row and pd.notna(row['rgb']):
        raw_signal_data = parse_raw_rgb(row['rgb'])
    elif 'g_signal' in row and pd.notna(row['g_signal']):
        raw_signal_data = parse_raw_rgb(row['g_signal'])

    if raw_signal_data is None:
        continue

    # Match with db.csv
    match_row = db_df[db_df['video'].str.contains(str(video_file), na=False)]
    if match_row.empty or 'rigidity' not in match_row.columns:
        continue

    rigidity_val = match_row.iloc[0]['rigidity']
    age_val = match_row.iloc[0]['age'] if 'age' in match_row.columns else 0
    sex_val = match_row.iloc[0]['sex_encoded'] if 'sex_encoded' in match_row.columns else 0

    if pd.isna(rigidity_val) or pd.isna(age_val):
        continue

    pos_sig = extract_pos_signal(raw_signal_data, target_len=FIXED_LENGTH)
    if pos_sig is not None:
        X_signals.append(pos_sig)
        demographics_list.append([float(age_val), float(sex_val)])
        y_rigidity.append(float(rigidity_val))

# Convert to arrays
X_pos = np.array(X_signals, dtype=np.float32)
X_demo = np.array(demographics_list, dtype=np.float32)
y = np.array(y_rigidity, dtype=np.float32)

# Normalize POS features and demographics independently
scaler_pos = StandardScaler()
X_pos_scaled = scaler_pos.fit_transform(X_pos)

scaler_demo = StandardScaler()
X_demo_scaled = scaler_demo.fit_transform(X_demo)

# Concatenate POS features (900 features) + Demographics (2 features) = 902 total inputs
X = np.hstack((X_pos_scaled, X_demo_scaled))

print(f"Dataset ready: {X.shape[0]} samples, {X.shape[1]} total features per sample (900 POS + 2 Demographics).")

# --- 4. K-FOLD CROSS VALIDATION ---
kf = KFold(n_splits=5, shuffle=True, random_state=42)
fold_mses = []
fold_rmses = []
fold_maes = []

for fold, (train_idx, test_idx) in enumerate(kf.split(X)):
    print(f"\n--- Training Fold {fold + 1}/5 ---")

    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
    model.fit(X_train, y_train)

    preds = model.predict(X_test)

    fold_mse = mean_squared_error(y_test, preds)
    fold_rmse = np.sqrt(fold_mse)
    fold_mae = mean_absolute_error(y_test, preds)

    fold_mses.append(fold_mse)
    fold_rmses.append(fold_rmse)
    fold_maes.append(fold_mae)

    print(f"Fold {fold + 1} MSE: {fold_mse:.4f} | RMSE: {fold_rmse:.4f} | MAE: {fold_mae:.4f}")

print("\n==================================================")
print("RANDOM FOREST (POS + AGE + SEX) RESULTS:")
print("==================================================")
print(f" -> Mean MSE:  {np.mean(fold_mses):.4f} (+/- {np.std(fold_mses):.4f})")
print(f" -> Mean RMSE: {np.mean(fold_rmses):.4f} (+/- {np.std(fold_rmses):.4f})")
print(f" -> Mean MAE:  {np.mean(fold_maes):.4f} (+/- {np.std(fold_maes):.4f})")
print("==================================================")

BMI+random forest+ CHROM + Sex and age

In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import LeaveOneGroupOut, cross_validate
from sklearn.preprocessing import StandardScaler, LabelEncoder

# --- CONFIGURATION ---
DATA_DIR = "/content/drive/MyDrive/Colab Notebooks/rPPG_Project/data"
ENHANCED_CSV = os.path.join(DATA_DIR, "enhanced_rppg_signals.csv")
DB_CSV = os.path.join(DATA_DIR, "db.csv")

enhanced_df = pd.read_csv(ENHANCED_CSV)
db_df = pd.read_csv(DB_CSV)

# Safely encode categorical 'sex' column if present
if 'sex' in db_df.columns:
    db_df['sex_encoded'] = LabelEncoder().fit_transform(db_df['sex'].astype(str))
else:
    db_df['sex_encoded'] = 0

def parse_signal(signal_input):
    """Safely convert signal strings back to numpy arrays."""
    if isinstance(signal_input, np.ndarray):
        return signal_input.astype(float)
    if isinstance(signal_input, list):
        return np.array(signal_input, dtype=float)
    if isinstance(signal_input, str):
        cleaned = signal_input.strip().replace("np.float64(", "").replace(")", "").replace("[", "").replace("]", "").replace("'", "").replace('"', "")
        if not cleaned:
            return np.array([])
        try:
            parts = [float(x.strip()) for x in cleaned.split(",") if x.strip()]
            if len(parts) > 0:
                return np.array(parts)
        except ValueError:
            pass
        try:
            parts = [float(x.strip()) for x in cleaned.split() if x.strip()]
            return np.array(parts)
        except ValueError:
            pass
    return np.array([], dtype=float)

def extract_morphology_features(signal, fps):
    """Extracts advanced waveform features from the CHROM rPPG signal."""
    sig = parse_signal(signal)
    if len(sig) < int(fps * 2):
        return None

    features = {
        'mean': np.mean(sig),
        'std': np.std(sig),
        'skew': pd.Series(sig).skew(),
        'kurtosis': pd.Series(sig).kurtosis(),
        'ptp_amplitude': np.ptp(sig),
        'rms': np.sqrt(np.mean(sig**2))
    }
    return features

# --- BUILD FEATURE MATRIX (X), TARGET (y), AND SUBJECT GROUPS (groups) ---
X_features = []
y_bmi = []
subject_groups = []

print("Extracting morphological features + demographics for LOSO BMI regression...")

for idx, row in enhanced_df.iterrows():
    video_file = row['video_file']
    chrom_sig = row['chrom_signal']
    fps = row['fps']

    # Match with db.csv to get target BMI, patient identifier, and demographics
    match_row = db_df[db_df['video'].str.contains(video_file, na=False)]
    if match_row.empty or 'bmi' not in match_row.columns:
        continue

    bmi_val = match_row.iloc[0]['bmi']
    if pd.isna(bmi_val):
        continue

    # Extract demographic values (default to 0 if missing)
    age_val = match_row.iloc[0]['age'] if 'age' in match_row.columns and pd.notna(match_row.iloc[0]['age']) else 0.0
    sex_val = match_row.iloc[0]['sex_encoded'] if 'sex_encoded' in match_row.columns else 0.0

    # Extract patient identifier
    subject_id = match_row.iloc[0].get('subject_id', row.get('subject_id', video_file))

    feats = extract_morphology_features(chrom_sig, fps)
    if feats is not None:
        # Combine waveform morphology features with demographics [age, sex]
        combined_features = list(feats.values()) + [float(age_val), float(sex_val)]

        X_features.append(combined_features)
        y_bmi.append(float(bmi_val))
        subject_groups.append(subject_id)

X = np.array(X_features)
y = np.array(y_bmi)
groups = np.array(subject_groups)

print(f"Dataset compiled: {X.shape[0]} valid samples across {len(np.unique(groups))} unique subjects ready for training.")
print(f"Feature vector size per sample: {X.shape[1]} (Morphology + Age + Sex)")

# --- TRAIN AND EVALUATE MODEL USING LOSO ---
if len(X) > 10:
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    model = RandomForestRegressor(n_estimators=100, random_state=42)

    # Leave-One-Group-Out cross-validation
    logo = LeaveOneGroupOut()

    scoring = {'mae': 'neg_mean_absolute_error', 'rmse': 'neg_root_mean_squared_error'}
    cv_results = cross_validate(model, X_scaled, y, groups=groups, cv=logo, scoring=scoring)

    mae_scores = -cv_results['test_mae']
    rmse_scores = -cv_results['test_rmse']

    print("\n==================================================")
    print("CHROM + DEMOGRAPHICS BMI PREDICTION (LOSO CV RESULTS):")
    print("==================================================")
    print(f" -> LOSO MAE:  {mae_scores.mean():.2f} (+/- {mae_scores.std():.2f}) kg/m²")
    print(f" -> LOSO RMSE: {rmse_scores.mean():.2f} (+/- {rmse_scores.std():.2f}) kg/m²")
    print("==================================================")
else:
    print("Not enough matching samples with valid BMI data.")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Extracting RPPG from the next 50 people (vectorization (or vectorized processing), and when applied specifically to sliding windows or frame sequences in computer vision and signal processing, it is often referred to as block-based processing or batch/array-level processing)

In [ ]:
import os
import numpy as np
import pandas as pd
from huggingface_hub import hf_hub_download, login

# --- 1. CONFIGURATION ---
HF_TOKEN = "YOUR_TOKEN"
login(token=HF_TOKEN)

REPO_ID = "wengziheng/mcd_rppg"
DATA_DIR = "/content/drive/MyDrive/Colab Notebooks/rPPG_Project/data"
db_csv_path = os.path.join(DATA_DIR, "db.csv")

# --- 2. LOAD METADATA & SELECT *NEXT* 50 PATIENTS (Patients 50 to 100) ---
df = pd.read_csv(db_csv_path)

id_col = 'patient_id' if 'patient_id' in df.columns else 'subject_id'
path_col = 'video' if 'video' in df.columns else 'video_path'

unique_patients = df[id_col].unique()
# Grab the next 50 unique patient IDs (indices 50 to 100)
selected_patients = unique_patients[50:100] if len(unique_patients) >= 100 else unique_patients[50:]
print(f"Targeting next batch: {len(selected_patients)} patients (Indices 50-{50+len(selected_patients)-1})...")

dev_df = df[df[id_col].isin(selected_patients)].dropna(subset=[path_col]).reset_index(drop=True)

print(f"Starting optimized batch download for {len(dev_df)} total video assets...\n")

downloaded_count = 0
failed_count = 0

for idx, row in dev_df.iterrows():
    relative_path = row[path_col].replace("\\", "/")
    full_path = os.path.join(DATA_DIR, relative_path)

    # Check if file already exists and is fully downloaded (> 1 MB)
    if os.path.exists(full_path) and (os.path.getsize(full_path) / (1024 * 1024)) > 1.0:
        print(f"[{idx+1}/{len(dev_df)}] ⏩ [EXISTS] {relative_path}")
        downloaded_count += 1
        continue

    print(f"[{idx+1}/{len(dev_df)}] ⏬ Downloading {relative_path}...")
    try:
        hf_hub_download(
            repo_id=REPO_ID,
            filename=relative_path,
            repo_type="dataset",
            local_dir=DATA_DIR,
            token=HF_TOKEN,
            force_download=False
        )
        downloaded_count += 1
        print(f"    ✔ Successfully downloaded!")
    except Exception as e:
        failed_count += 1
        print(f"    ✖ Failed: {e}")

print("\n==========================================")
print(f"BATCH DOWNLOAD SUMMARY:")
print(f" -> Successfully On Disk: {downloaded_count} / {len(dev_df)}")
print(f" -> Failed Downloads:     {failed_count}")
print("==========================================")

Check to see how many extraction there are in data

In [ ]:
import os
import pandas as pd

DATA_DIR = "/content/drive/MyDrive/Colab Notebooks/rPPG_Project/data"
db_csv_path = os.path.join(DATA_DIR, "db.csv")

# Load metadata
df = pd.read_csv(db_csv_path)
path_col = 'video' if 'video' in df.columns else 'video_path'
id_col = 'patient_id' if 'patient_id' in df.columns else 'subject_id'

valid_df = df.dropna(subset=[path_col]).copy()
total_patients = valid_df[id_col].nunique()
total_videos = len(valid_df)

# Check how many are actually on disk
downloaded_videos = 0
missing_videos = []

for idx, row in valid_df.iterrows():
    rel_path = str(row[path_col]).replace("\\", "/")
    full_path = os.path.join(DATA_DIR, rel_path)

    if os.path.exists(full_path) and (os.path.getsize(full_path) / (1024 * 1024)) > 1.0:
        downloaded_videos += 1
    else:
        missing_videos.append(rel_path)

print("==========================================")
print(f"DATASET AUDIT SUMMARY:")
print(f" -> Total Unique Patients in Metadata: {total_patients}")
print(f" -> Total Video Files in Metadata:     {total_videos}")
print(f" -> Files Successfully Downloaded:     {downloaded_videos}")
print(f" -> Files Remaining / Missing:         {len(missing_videos)}")
print("==========================================")

if len(missing_videos) > 0:
    print(f"First few remaining files to pull:")
    for f in missing_videos[:5]:
        print(f" - {f}")
else:
    print("🎉 All video assets from the dataset have been fully downloaded!")

In [ ]:
!pip uninstall -y opencv-python opencv-contrib-python
!pip install opencv-contrib-python

Extracting the RGB signal

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm

# --- 1. CONFIGURATION ---
DATA_DIR = "/content/drive/MyDrive/Colab Notebooks/rPPG_Project/data"
OUTPUT_RPPG_CSV = os.path.join(DATA_DIR, "raw_rppg_signals.csv")
VIDEO_EXTENSIONS = ('.mp4', '.avi', '.mov', '.mkv', '.webm')

print("Scanning folder for available videos on disk...")

# Find all actual video files inside DATA_DIR and its subdirectories
disk_videos = {} # filename -> full_path
for root, dirs, files in os.walk(DATA_DIR):
    for file in files:
        if file.lower().endswith(VIDEO_EXTENSIONS):
            filename = file
            full_path = os.path.join(root, file)
            disk_videos[filename] = full_path

print(f" -> Found {len(disk_videos)} total video files on disk.")

# --- 2. CHECK ALREADY EXTRACTED VIDEOS ---
completed_videos = set()
if os.path.exists(OUTPUT_RPPG_CSV):
    existing_df = pd.read_csv(OUTPUT_RPPG_CSV)
    if 'video_file' in existing_df.columns:
        completed_videos = set(existing_df['video_file'].dropna().unique())

print(f" -> Found {len(completed_videos)} already extracted videos in raw_rppg_signals.csv.")

# Determine which files on disk have NOT been extracted yet
pending_videos = {fname: fpath for fname, fpath in disk_videos.items() if fname not in completed_videos}

print(f" -> Remaining videos to extract: {len(pending_videos)}")

# --- 3. BLOCK-BASED VECTORIZED EXTRACTION ---
def extract_rppg_block_based(full_path, batch_size=300):
    cap = cv2.VideoCapture(full_path, cv2.CAP_FFMPEG)
    if not cap.isOpened():
        return None, None, None, 0

    fps = cap.get(cv2.CAP_PROP_FPS)
    if not fps or fps <= 0 or np.isnan(fps):
        fps = 29.90

    red_channel, green_channel, blue_channel = [], [], []

    while cap.isOpened():
        frame_batch = []
        for _ in range(batch_size):
            ret, frame = cap.read()
            if not ret:
                break
            frame_batch.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))

        if not frame_batch:
            break

        block_arr = np.array(frame_batch, dtype=np.float32)
        _, h, w, _ = block_arr.shape

        roi_block = block_arr[:, int(h*0.15):int(h*0.65), int(w*0.3):int(w*0.7), :]
        block_means = np.mean(roi_block, axis=(1, 2))

        red_channel.extend(block_means[:, 0])
        green_channel.extend(block_means[:, 1])
        blue_channel.extend(block_means[:, 2])

    cap.release()
    if len(red_channel) == 0:
        return None, None, None, 0

    return red_channel, green_channel, blue_channel, fps

# --- 4. LIVE INCREMENTAL EXTRACTION & IMMEDIATE SAVE ---
success_count = 0
skipped_count = 0

if len(pending_videos) == 0:
    print("\n🎉 All available videos on disk have already been extracted and saved!")
else:
    with tqdm(pending_videos.items(), desc="Extracting & Saving Live") as pbar:
        for filename, full_path in pbar:
            # Try to infer a patient ID from the filename if possible (e.g., matching digits), or default to a generic tag
            # Adjust this parsing logic if your filenames follow a specific naming convention
            patient_id = "".join([char for char in filename if char.isdigit()])
            patient_id = int(patient_id[:4]) if len(patient_id) >= 4 else filename

            # Extract RGB signals
            r, g, b, fps = extract_rppg_block_based(full_path)

            if r is not None and len(r) > 0:
                single_record_df = pd.DataFrame([{
                    'patient_id': patient_id,
                    'video_file': filename,
                    'fps': fps,
                    'total_frames': len(r),
                    'r_signal': r,
                    'g_signal': g,
                    'b_signal': b
                }])

                # --- IMMEDIATE APPEND TO CSV ---
                file_exists = os.path.exists(OUTPUT_RPPG_CSV)
                single_record_df.to_csv(
                    OUTPUT_RPPG_CSV,
                    mode='a',
                    index=False,
                    header=not file_exists
                )

                success_count += 1
            else:
                skipped_count += 1

            pbar.set_postfix(file=filename, saved=success_count, skipped=skipped_count)

print("\n==========================================")
print("EXTRACTION & LIVE SAVE COMPLETE!")
print(f" -> Newly Extracted & Saved: {success_count} videos")
print(f" -> Saved directly to:       {OUTPUT_RPPG_CSV}")
print("==========================================")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import pandas as pd

DATA_DIR = "/content/drive/MyDrive/Colab Notebooks/rPPG_Project/data"
OUTPUT_RPPG_CSV = os.path.join(DATA_DIR, "raw_rppg_signals.csv")

if os.path.exists(OUTPUT_RPPG_CSV):
    df = pd.read_csv(OUTPUT_RPPG_CSV)
    id_col = 'patient_id' if 'patient_id' in df.columns else 'subject_id'

    unique_patients = df[id_col].nunique()
    total_records = len(df)

    print(f"📊 Extraction Status Report:")
    print(f" -> Total video records saved: {total_records}")
    print(f" -> Unique patients completed: {unique_patients}")
    print(f" -> List of extracted patient IDs:\n{df[id_col].unique().tolist()}")
else:
    print(f"⚠️ No extracted file found yet at: {OUTPUT_RPPG_CSV}")

In [ ]:
import os
import numpy as np
import pandas as pd
from tqdm import tqdm

# --- 1. CONFIGURATION ---
DATA_DIR = "/content/drive/MyDrive/Colab Notebooks/rPPG_Project/data"
RAW_RPPG_CSV = os.path.join(DATA_DIR, "raw_rppg_signals.csv")
ENHANCED_RPPG_CSV = os.path.join(DATA_DIR, "enhanced_rppg_signals.csv")

print("Loading raw rPPG signals and checking existing progress...")
if not os.path.exists(RAW_RPPG_CSV):
    raise FileNotFoundError(f"Raw signals file not found at: {RAW_RPPG_CSV}")

raw_df = pd.read_csv(RAW_RPPG_CSV)

# Check what has already been processed in enhanced_rppg_signals.csv
completed_videos = set()
if os.path.exists(ENHANCED_RPPG_CSV):
    existing_enhanced = pd.read_csv(ENHANCED_RPPG_CSV)
    if 'video_file' in existing_enhanced.columns:
        completed_videos = set(existing_enhanced['video_file'].dropna().unique())

print(f" -> Found {len(completed_videos)} already filtered videos on disk.")

# --- 2. CHROM FILTER FUNCTION ---
def chrom_filter(r, g, b, fps=30.0):
    """
    Applies the De Haan & Jeanne (2013) CHROM algorithm for rPPG extraction.
    """
    # Convert string representation of lists back to numpy arrays if stored as strings
    if isinstance(r, str):
        r = np.array(eval(r), dtype=np.float32)
    if isinstance(g, str):
        g = np.array(eval(g), dtype=np.float32)
    if isinstance(b, str):
        b = np.array(eval(b), dtype=np.float32)

    r = np.array(r, dtype=np.float32)
    g = np.array(g, dtype=np.float32)
    b = np.array(b, dtype=np.float32)

    if len(r) < 30:
        return np.zeros_like(r)

    # 1. Normalize (spatial-temporal mean scaling or simple dividing by mean)
    # Standard CHROM color normalization
    mean_r, mean_g, mean_b = np.mean(r), np.mean(g), np.mean(b)
    if mean_r == 0 or mean_g == 0 or mean_b == 0:
        return np.zeros_like(r)

    Rn = r / mean_r
    Gn = g / mean_g
    Bn = b / mean_b

    # 2. Projection equations for CHROM
    Xs = 3.0 * Rn - 2.0 * Gn
    Ys = 1.5 * Rn + Gn - 1.5 * Bn

    # 3. Bandpass filtering (optional/simplified sliding window or standard detrending)
    # Simple bandpass approximation via temporal windowing / detrending can be added here,
    # but core chrominance signal projection is Xs - (sigma * Ys)
    # Standard tuning parameter alpha = std(Xs) / std(Ys)
    std_x = np.std(Xs)
    std_y = np.std(Ys)

    alpha = std_x / std_y if std_y > 0 else 1.0
    s = Xs - alpha * Ys

    # Detrend / Normalize output waveform
    s = s - np.mean(s)
    std_s = np.std(s)
    if std_s > 0:
        s = s / std_s

    return s.tolist()

# --- 3. BATCH PROCESSING & INCREMENTAL SAVE ---
success_count = 0
skipped_count = 0

with tqdm(raw_df.iterrows(), total=len(raw_df), desc="Applying CHROM Filter") as pbar:
    for idx, row in pbar:
        filename = row['video_file']
        patient_id = row['patient_id']
        fps = row.get('fps', 30.0)

        # Skip if already processed in enhanced CSV
        if filename in completed_videos:
            skipped_count += 1
            continue

        r_sig = row['r_signal']
        g_sig = row['g_signal']
        b_sig = row['b_signal']

        # Apply CHROM filter
        filtered_signal = chrom_filter(r_sig, g_sig, b_sig, fps=fps)

        if filtered_signal is not None and len(filtered_signal) > 0:
            enhanced_record = pd.DataFrame([{
                'patient_id': patient_id,
                'video_file': filename,
                'fps': fps,
                'total_frames': len(filtered_signal),
                'filtered_rppg': filtered_signal
            }])

            # Immediate incremental append
            file_exists = os.path.exists(ENHANCED_RPPG_CSV)
            enhanced_record.to_csv(
                ENHANCED_RPPG_CSV,
                mode='a',
                index=False,
                header=not file_exists
            )

            completed_videos.add(filename)
            success_count += 1
        else:
            skipped_count += 1

        pbar.set_postfix(patient=str(patient_id), filtered=success_count, skipped=skipped_count)

print("\n==========================================")
print("CHROM FILTERING & ENHANCED SAVE COMPLETE!")
print(f" -> Newly Filtered & Saved: {success_count} records")
print(f" -> Saved directly to:        {ENHANCED_RPPG_CSV}")
print("==========================================")

Arterial Stiffness + Chrom + 1DCNN+ LOSO

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error

# --- CONFIGURATION ---
DATA_DIR = "/content/drive/MyDrive/Colab Notebooks/rPPG_Project/data"
ENHANCED_CSV = os.path.join(DATA_DIR, "enhanced_rppg_signals.csv")
DB_CSV = os.path.join(DATA_DIR, "db.csv")
TARGET_PATIENT_LIMIT = 100

enhanced_df = pd.read_csv(ENHANCED_CSV)
db_df = pd.read_csv(DB_CSV)
FIXED_LENGTH = 900  # 30 seconds at 30 fps

# --- RESTRICT TO 100 UNIQUE PATIENTS FROM ENHANCED CSV ---
if 'patient_id' in enhanced_df.columns:
    unique_patients = enhanced_df['patient_id'].dropna().unique()[:TARGET_PATIENT_LIMIT]
    enhanced_df = enhanced_df[enhanced_df['patient_id'].isin(unique_patients)].reset_index(drop=True)
    print(f" -> Restricted enhanced data to {len(unique_patients)} unique patients.")
else:
    # Fallback if filtered by video count / row slice
    enhanced_df = enhanced_df.head(TARGET_PATIENT_LIMIT)

# Safely encode categorical 'sex' column if present
if 'sex' in db_df.columns:
    db_df['sex_encoded'] = LabelEncoder().fit_transform(db_df['sex'].astype(str))
else:
    db_df['sex_encoded'] = 0

def parse_signal(signal_input):
    """Safely convert signal strings back to numpy arrays and pad/truncate to fixed length."""
    if isinstance(signal_input, np.ndarray):
        sig = signal_input.astype(float)
    elif isinstance(signal_input, list):
        sig = np.array(signal_input, dtype=float)
    elif isinstance(signal_input, str):
        cleaned = signal_input.strip().replace("np.float64(", "").replace(")", "").replace("[", "").replace("]", "").replace("'", "").replace('"', "")
        if not cleaned:
            return None
        try:
            parts = [float(x.strip()) for x in cleaned.split(",") if x.strip()]
            sig = np.array(parts, dtype=float)
        except ValueError:
            try:
                parts = [float(x.strip()) for x in cleaned.split() if x.strip()]
                sig = np.array(parts, dtype=float)
            except ValueError:
                return None
    else:
        return None

    if len(sig) == 0:
        return None

    if len(sig) < FIXED_LENGTH:
        sig = np.pad(sig, (0, FIXED_LENGTH - len(sig)), 'edge')
    else:
        sig = sig[:FIXED_LENGTH]
    return sig

# --- BUILD TENSORS & DEMOGRAPHICS ---
X_signals = []
demographics_list = []
y_rigidity = []

print("Preparing CHROM signals and demographics for Arterial Stiffness (Rigidity) 1D-CNN...")
for idx, row in enhanced_df.iterrows():
    video_file = row['video_file']
    chrom_sig = row['chrom_signal']

    match_row = db_df[db_df['video'].str.contains(video_file, na=False)]
    if match_row.empty or 'rigidity' not in match_row.columns:
        continue

    rigidity_val = match_row.iloc[0]['rigidity']
    if pd.isna(rigidity_val):
        continue

    # Extract age and sex (default to 0 if missing)
    age_val = match_row.iloc[0]['age'] if 'age' in match_row.columns and pd.notna(match_row.iloc[0]['age']) else 0.0
    sex_val = match_row.iloc[0]['sex_encoded'] if 'sex_encoded' in match_row.columns else 0.0

    processed_sig = parse_signal(chrom_sig)
    if processed_sig is not None:
        X_signals.append(processed_sig)
        demographics_list.append([float(age_val), float(sex_val)])
        y_rigidity.append(float(rigidity_val))

X_sig_arr = np.array(X_signals, dtype=np.float32)
X_demo_arr = np.array(demographics_list, dtype=np.float32)
y = np.array(y_rigidity, dtype=np.float32)

# Normalize signals and demographics
scaler_sig = StandardScaler()
X_sig_scaled = scaler_sig.fit_transform(X_sig_arr)

scaler_demo = StandardScaler()
X_demo_scaled = scaler_demo.fit_transform(X_demo_arr)

# Reshape signals for PyTorch Conv1D: [Batch, Channels=1, Length=900]
X_sig_scaled = np.expand_dims(X_sig_scaled, axis=1)

print(f"Dataset ready for CNN: {X_sig_scaled.shape[0]} valid samples across 100-patient target pool.")

# --- HYBRID 1D-CNN ARCHITECTURE (WAVEFORM + DEMOGRAPHICS) ---
class Hybrid_RPPG_Rigidity_Net(nn.Module):
    def __init__(self):
        super(Hybrid_RPPG_Rigidity_Net, self).__init__()
        self.features = nn.Sequential(
            nn.Conv1d(1, 16, kernel_size=7, stride=2, padding=3),
            nn.BatchNorm1d(16),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(16, 32, kernel_size=5, stride=1, padding=2),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(32, 64, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1)  # Output shape: [Batch, 64, 1]
        )

        # Fully connected layers take CNN output (64) + demographics (2) = 66 features
        self.regressor = nn.Sequential(
            nn.Linear(64 + 2, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 1)
        )

    def forward(self, x_sig, x_demo):
        x_feat = self.features(x_sig)
        x_feat = x_feat.view(x_feat.size(0), -1)  # Flatten CNN features

        # Concatenate waveform features with age and sex metadata
        combined = torch.cat((x_feat, x_demo), dim=1)

        out = self.regressor(combined)
        return out.squeeze()

# --- K-FOLD CROSS VALIDATION FOR HYBRID CNN ---
kf = KFold(n_splits=5, shuffle=True, random_state=42)
fold_maes = []
fold_rmses = []

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on device: {device}")

for fold, (train_idx, test_idx) in enumerate(kf.split(X_sig_scaled)):
    print(f"\n--- Training Fold {fold + 1}/5 ---")

    X_train_sig = torch.tensor(X_sig_scaled[train_idx], dtype=torch.float32)
    X_train_demo = torch.tensor(X_demo_scaled[train_idx], dtype=torch.float32)
    y_train = torch.tensor(y[train_idx], dtype=torch.float32)

    X_test_sig = torch.tensor(X_sig_scaled[test_idx], dtype=torch.float32)
    X_test_demo = torch.tensor(X_demo_scaled[test_idx], dtype=torch.float32)
    y_test = torch.tensor(y[test_idx], dtype=torch.float32)

    train_loader = DataLoader(TensorDataset(X_train_sig, X_train_demo, y_train), batch_size=16, shuffle=True)

    model = Hybrid_RPPG_Rigidity_Net().to(device)
    criterion = nn.L1Loss()  # MAE loss
    optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)

    # Training loop
    model.train()
    for epoch in range(25):
        for batch_sig, batch_demo, batch_y in train_loader:
            batch_sig, batch_demo, batch_y = batch_sig.to(device), batch_demo.to(device), batch_y.to(device)
            optimizer.zero_grad()
            outputs = model(batch_sig, batch_demo)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()

    # Evaluation
    model.eval()
    with torch.no_grad():
        test_sig_dev = X_test_sig.to(device)
        test_demo_dev = X_test_demo.to(device)
        test_y_dev = y_test.to(device)

        preds = model(test_sig_dev, test_demo_dev)

        fold_mae = mean_absolute_error(test_y_dev.cpu().numpy(), preds.cpu().numpy())
        fold_rmse = np.sqrt(mean_squared_error(test_y_dev.cpu().numpy(), preds.cpu().numpy()))

        fold_maes.append(fold_mae)
        fold_rmses.append(fold_rmse)
        print(f"Fold {fold + 1} MAE: {fold_mae:.4f} | RMSE: {fold_rmse:.4f}")

print("\n==================================================")
print("HYBRID 1D-CNN (100 PATIENTS: CHROM + AGE + SEX) RESULTS:")
print("==================================================")
print(f" -> Mean MAE:  {np.mean(fold_maes):.4f} (+/- {np.std(fold_maes):.4f})")
print(f" -> Mean RMSE: {np.mean(fold_rmses):.4f} (+/- {np.std(fold_rmses):.4f})")

BMI+ CHROM + 1DCNN+ Random Forest + LOSO

In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, cross_val_score
from sklearn.preprocessing import StandardScaler

# --- CONFIGURATION ---
DATA_DIR = "/content/drive/MyDrive/Colab Notebooks/rPPG_Project/data"
ENHANCED_CSV = os.path.join(DATA_DIR, "enhanced_rppg_signals.csv")
DB_CSV = os.path.join(DATA_DIR, "db.csv")
TARGET_PATIENT_LIMIT = 100

enhanced_df = pd.read_csv(ENHANCED_CSV)
db_df = pd.read_csv(DB_CSV)

# --- RESTRICT TO 100 UNIQUE PATIENTS FROM ENHANCED CSV ---
#if 'patient_id' in enhanced_df.columns:
   # unique_patients = enhanced_df['patient_id'].dropna().unique()[:TARGET_PATIENT_LIMIT]
   # enhanced_df = enhanced_df[enhanced_df['patient_id'].isin(unique_patients)].reset_index(drop=True)
   # print(f" -> Restricted enhanced data to {len(unique_patients)} unique patients.")
#else:
    #enhanced_df = enhanced_df.head(TARGET_PATIENT_LIMIT)

def parse_signal(signal_input):
    """Safely convert signal strings back to numpy arrays."""
    if isinstance(signal_input, np.ndarray):
        return signal_input.astype(float)
    if isinstance(signal_input, list):
        return np.array(signal_input, dtype=float)
    if isinstance(signal_input, str):
        cleaned = signal_input.strip().replace("np.float64(", "").replace(")", "").replace("[", "").replace("]", "").replace("'", "").replace('"', "")
        if not cleaned:
            return np.array([])
        try:
            parts = [float(x.strip()) for x in cleaned.split(",") if x.strip()]
            if len(parts) > 0:
                return np.array(parts)
        except ValueError:
            pass
        try:
            parts = [float(x.strip()) for x in cleaned.split() if x.strip()]
            return np.array(parts)
        except ValueError:
            pass
    return np.array([], dtype=float)

def extract_morphology_features(signal, fps):
    """Extracts advanced waveform features from the CHROM rPPG signal."""
    sig = parse_signal(signal)
    if len(sig) < int(fps * 2):
        return None

    # Statistical & shape features capturing pulse contour and energy
    features = {
        'mean': np.mean(sig),
        'std': np.std(sig),
        'skew': pd.Series(sig).skew(),
        'kurtosis': pd.Series(sig).kurtosis(),
        'ptp_amplitude': np.ptp(sig),
        'rms': np.sqrt(np.mean(sig**2))
    }
    return features

# --- BUILD FEATURE MATRIX (X) AND TARGET (y) ---
X_features = []
y_bmi = []

print("Extracting morphological features for BMI regression (100-patient pool)...")

for idx, row in enhanced_df.iterrows():
    video_file = row['video_file']
    chrom_sig = row['chrom_signal']
    fps = row['fps']

    # Match with db.csv to get the target BMI
    match_row = db_df[db_df['video'].str.contains(video_file, na=False)]
    if match_row.empty or 'bmi' not in match_row.columns:
        continue

    bmi_val = match_row.iloc[0]['bmi']
    if pd.isna(bmi_val):
        continue

    feats = extract_morphology_features(chrom_sig, fps)
    if feats is not None:
        X_features.append(list(feats.values()))
        y_bmi.append(float(bmi_val))

X = np.array(X_features)
y = np.array(y_bmi)

print(f"Dataset compiled: {X.shape[0]} valid samples ready for training.")

# --- TRAIN AND EVALUATE MODEL ---
if len(X) > 10:
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    model = RandomForestRegressor(n_estimators=100, random_state=42)

    # 5-fold cross-validation evaluating Mean Absolute Error (MAE) in kg/m²
    cv = KFold(n_splits=5, shuffle=True, random_state=42)
    mae_scores = -cross_val_score(model, X_scaled, y, cv=cv, scoring='neg_mean_absolute_error')

    print("\n==========================================")
    print("CHROM-BASED BMI PREDICTION RESULTS:")
    print("==========================================")
    print(f" -> Cross-Validation MAE: {mae_scores.mean():.2f} (+/- {mae_scores.std():.2f}) kg/m²")
    print("==========================================")
else:
    print("Not enough matching samples with valid BMI data.")

In [ ]:
print(f"📊 Diagnostic Check:")
print(f" -> Total rows loaded in X: {len(X)}")
print(f" -> Unique patients in enhanced_df: {enhanced_df['patient_id'].nunique() if 'patient_id' in enhanced_df.columns else 'N/A'}")

1DCNN + BMI

In [ ]:
import os
import ast
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error

# --- 1. CONFIGURATION ---
DATA_DIR = "/content/drive/MyDrive/Colab Notebooks/rPPG_Project/data"
ENHANCED_CSV = os.path.join(DATA_DIR, "enhanced_rppg_signals.csv")
DB_CSV = os.path.join(DATA_DIR, "db.csv")
TARGET_PATIENT_LIMIT = 100
FIXED_LENGTH = 900  # 30 seconds at 30 fps

print("Loading datasets...")
enhanced_df = pd.read_csv(ENHANCED_CSV)
db_df = pd.read_csv(DB_CSV)

# --- 2. RESTRICT TO 100 UNIQUE PATIENTS ---
if 'patient_id' in enhanced_df.columns:
    unique_patients = enhanced_df['patient_id'].dropna().unique()[:TARGET_PATIENT_LIMIT]
    enhanced_df = enhanced_df[enhanced_df['patient_id'].isin(unique_patients)].reset_index(drop=True)
    print(f" -> Restricted enhanced data to {len(unique_patients)} unique patients ({len(enhanced_df)} total video rows).")
else:
    enhanced_df = enhanced_df.head(TARGET_PATIENT_LIMIT)

# Safely encode categorical 'sex' column if present
if 'sex' in db_df.columns:
    db_df['sex_encoded'] = LabelEncoder().fit_transform(db_df['sex'].astype(str))
else:
    db_df['sex_encoded'] = 0

def parse_signal(signal_input):
    """Safely convert signal strings back to numpy arrays and pad/truncate."""
    if isinstance(signal_input, np.ndarray):
        sig = signal_input.astype(float)
    elif isinstance(signal_input, list):
        sig = np.array(signal_input, dtype=float)
    elif isinstance(signal_input, str):
        cleaned = signal_input.strip().replace("np.float64(", "").replace(")", "").replace("[", "").replace("]", "").replace("'", "").replace('"', "")
        if not cleaned:
            return None
        try:
            parts = [float(x.strip()) for x in cleaned.split(",") if x.strip()]
            sig = np.array(parts, dtype=float)
        except ValueError:
            try:
                parts = [float(x.strip()) for x in cleaned.split() if x.strip()]
                sig = np.array(parts, dtype=float)
            except ValueError:
                return None
    else:
        return None

    if len(sig) == 0:
        return None

    if len(sig) < FIXED_LENGTH:
        sig = np.pad(sig, (0, FIXED_LENGTH - len(sig)), 'edge')
    else:
        sig = sig[:FIXED_LENGTH]
    return sig

# --- 3. BUILD TENSORS, DEMOGRAPHICS & GROUPS ---
X_signals = []
demographics_list = []
y_target = []
patient_groups = []

print("Preparing CHROM signals, demographics, and patient grouping...")
for idx, row in enhanced_df.iterrows():
    video_file = row['video_file']
    chrom_sig = row['chrom_signal']
    patient_id = row['patient_id']

    # Match with db.csv to get target rigidity and demographics
    match_row = db_df[db_df['video'].str.contains(video_file, na=False)]
    if match_row.empty or 'rigidity' not in match_row.columns:
        continue

    rigidity_val = match_row.iloc[0]['rigidity']
    if pd.isna(rigidity_val):
        continue

    age_val = match_row.iloc[0]['age'] if 'age' in match_row.columns and pd.notna(match_row.iloc[0]['age']) else 0.0
    sex_val = match_row.iloc[0]['sex_encoded'] if 'sex_encoded' in match_row.columns else 0.0

    processed_sig = parse_signal(chrom_sig)
    if processed_sig is not None:
        X_signals.append(processed_sig)
        demographics_list.append([float(age_val), float(sex_val)])
        y_target.append(float(rigidity_val))
        patient_groups.append(patient_id)

X_sig_arr = np.array(X_signals, dtype=np.float32)
X_demo_arr = np.array(demographics_list, dtype=np.float32)
y = np.array(y_target, dtype=np.float32)
groups = np.array(patient_groups)

# Normalize signals and demographics
scaler_sig = StandardScaler()
X_sig_scaled = scaler_sig.fit_transform(X_sig_arr)

scaler_demo = StandardScaler()
X_demo_scaled = scaler_demo.fit_transform(X_demo_arr)

# Reshape signals for PyTorch Conv1d: [Batch, Channels=1, Length=900]
X_sig_scaled = np.expand_dims(X_sig_scaled, axis=1)

print(f"Dataset compiled: {X_sig_scaled.shape[0]} valid samples across {len(np.unique(groups))} unique patients.")

# --- 4. HYBRID 1D-CNN ARCHITECTURE ---
class Hybrid_RPPG_Rigidity_Net(nn.Module):
    def __init__(self):
        super(Hybrid_RPPG_Rigidity_Net, self).__init__()
        self.features = nn.Sequential(
            nn.Conv1d(1, 16, kernel_size=7, stride=2, padding=3),
            nn.BatchNorm1d(16),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(16, 32, kernel_size=5, stride=1, padding=2),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(32, 64, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1)  # Output shape: [Batch, 64, 1]
        )

        # Fully connected layers: CNN output (64) + demographics (2) = 66 features
        self.regressor = nn.Sequential(
            nn.Linear(64 + 2, 32),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(32, 1)
        )

    def forward(self, x_sig, x_demo):
        x_feat = self.features(x_sig)
        x_feat = x_feat.view(x_feat.size(0), -1)  # Flatten

        combined = torch.cat((x_feat, x_demo), dim=1)
        out = self.regressor(combined)
        return out.squeeze()

# --- 5. GROUP 5-FOLD CROSS VALIDATION ---
gkf = GroupKFold(n_splits=5)
fold_maes = []
fold_rmses = []

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on device: {device}")

for fold, (train_idx, test_idx) in enumerate(gkf.split(X_sig_scaled, y, groups=groups)):
    print(f"\n--- Training Fold {fold + 1}/5 (Patient-Independent Split) ---")

    X_train_sig = torch.tensor(X_sig_scaled[train_idx], dtype=torch.float32)
    X_train_demo = torch.tensor(X_demo_scaled[train_idx], dtype=torch.float32)
    y_train = torch.tensor(y[train_idx], dtype=torch.float32)

    X_test_sig = torch.tensor(X_sig_scaled[test_idx], dtype=torch.float32)
    X_test_demo = torch.tensor(X_demo_scaled[test_idx], dtype=torch.float32)
    y_test = torch.tensor(y[test_idx], dtype=torch.float32)

    train_loader = DataLoader(TensorDataset(X_train_sig, X_train_demo, y_train), batch_size=16, shuffle=True)

    model = Hybrid_RPPG_Rigidity_Net().to(device)
    criterion = nn.L1Loss()  # Mean Absolute Error loss
    optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)

    # Training loop
    model.train()
    for epoch in range(30):
        for batch_sig, batch_demo, batch_y in train_loader:
            batch_sig, batch_demo, batch_y = batch_sig.to(device), batch_demo.to(device), batch_y.to(device)
            optimizer.zero_grad()
            outputs = model(batch_sig, batch_demo)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()

    # Evaluation
    model.eval()
    with torch.no_grad():
        test_sig_dev = X_test_sig.to(device)
        test_demo_dev = X_test_demo.to(device)
        test_y_dev = y_test.to(device)

        preds = model(test_sig_dev, test_demo_dev)

        fold_mae = mean_absolute_error(test_y_dev.cpu().numpy(), preds.cpu().numpy())
        fold_rmse = np.sqrt(mean_squared_error(test_y_dev.cpu().numpy(), preds.cpu().numpy()))

        fold_maes.append(fold_mae)
        fold_rmses.append(fold_rmse)
        print(f" -> Fold {fold + 1} MAE: {fold_mae:.4f} | RMSE: {fold_rmse:.4f}")

print("\n==================================================")
print("HYBRID 1D-CNN (100 PATIENTS GROUPKFOLD) RESULTS:")
print("==================================================")
print(f" -> Mean MAE:  {np.mean(fold_maes):.4f} (+/- {np.std(fold_maes):.4f})")
print(f" -> Mean RMSE: {np.mean(fold_rmses):.4f} (+/- {np.std(fold_rmses):.4f})")
print("==================================================")

Download the rest of the data

In [ ]:
import os
import numpy as np
import pandas as pd
from huggingface_hub import hf_hub_download, login

# --- 1. CONFIGURATION ---
HF_TOKEN = "YOUR_TOKEN"
login(token=HF_TOKEN)

REPO_ID = "wengziheng/mcd_rppg"
DATA_DIR = "/content/drive/MyDrive/Colab Notebooks/rPPG_Project/data"
db_csv_path = os.path.join(DATA_DIR, "db.csv")

# --- 2. LOAD METADATA FOR ALL PATIENTS & VIDEOS ---
df = pd.read_csv(db_csv_path)

id_col = 'patient_id' if 'patient_id' in df.columns else 'subject_id'
path_col = 'video' if 'video' in df.columns else 'video_path'

# Use all rows with a valid video path across the entire dataset
dev_df = df.dropna(subset=[path_col]).reset_index(drop=True)

print(f"Checking all {len(dev_df)} total video assets across all patients in db.csv...")
print(f"Files already on disk (>1MB) will be automatically skipped.\n")

downloaded_count = 0
skipped_count = 0
failed_count = 0

for idx, row in dev_df.iterrows():
    relative_path = row[path_col].replace("\\", "/")
    full_path = os.path.join(DATA_DIR, relative_path)

    # Check if file already exists and is fully downloaded (> 1 MB)
    if os.path.exists(full_path) and (os.path.getsize(full_path) / (1024 * 1024)) > 1.0:
        skipped_count += 1
        # Print sparingly or every 50 files to avoid cluttering Colab logs
        if (idx + 1) % 50 == 0 or idx == 0:
            print(f"[{idx+1}/{len(dev_df)}] ⏩ [ALREADY EXISTS] Skipping downloaded files...")
        continue

    print(f"[{idx+1}/{len(dev_df)}] ⏬ Downloading missing asset: {relative_path}...")
    try:
        hf_hub_download(
            repo_id=REPO_ID,
            filename=relative_path,
            repo_type="dataset",
            local_dir=DATA_DIR,
            token=HF_TOKEN,
            force_download=False
        )
        downloaded_count += 1
        print(f"    ✔ Successfully downloaded!")
    except Exception as e:
        failed_count += 1
        print(f"    ✖ Failed: {e}")

print("\n==========================================")
print(f"UNIVERSAL DOWNLOAD SUMMARY:")
print(f" -> Already on Disk (Skipped): {skipped_count}")
print(f" -> Newly Downloaded:          {downloaded_count}")
print(f" -> Failed Downloads:          {failed_count}")
print(f" -> Total Accounted For:       {skipped_count + downloaded_count} / {len(dev_df)}")
print("==========================================")

Remainder of the extracted_ rppg_signals

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm

# --- 1. CONFIGURATION ---
DATA_DIR = "/content/drive/MyDrive/Colab Notebooks/rPPG_Project/data"
OUTPUT_RPPG_CSV = os.path.join(DATA_DIR, "raw_rppg_signals.csv")
VIDEO_EXTENSIONS = ('.mp4', '.avi', '.mov', '.mkv', '.webm')

print("Scanning folder for available videos on disk...")

disk_videos = {}
for root, dirs, files in os.walk(DATA_DIR):
    for file in files:
        if file.lower().endswith(VIDEO_EXTENSIONS):
            disk_videos[file] = os.path.join(root, file)

print(f" -> Found {len(disk_videos)} total video files on disk.")

# --- 2. ROBUST CHECK FOR COMPLETED VIDEOS ---
completed_videos = set()
if os.path.exists(OUTPUT_RPPG_CSV):
    try:
        existing_df = pd.read_csv(OUTPUT_RPPG_CSV, engine='python', on_bad_lines='skip')
        if 'video_file' in existing_df.columns:
            completed_videos = set(existing_df['video_file'].dropna().unique())
        print(f" -> Found {len(completed_videos)} already extracted videos in raw_rppg_signals.csv.")
    except Exception as e:
        print(f"⚠️ Warning: Could not fully parse existing CSV ({e}). Proceeding safely.")

pending_videos = {fname: fpath for fname, fpath in disk_videos.items() if fname not in completed_videos}
print(f" -> Remaining videos to extract: {len(pending_videos)}")

# --- 3. HIGH-SPEED OPTIMIZED EXTRACTION ---
def extract_rppg_fast(full_path, batch_size=600):
    cap = cv2.VideoCapture(full_path, cv2.CAP_FFMPEG)
    if not cap.isOpened():
        return None, None, None, 0

    fps = cap.get(cv2.CAP_PROP_FPS)
    if not fps or fps <= 0 or np.isnan(fps):
        fps = 29.90

    red_list, green_list, blue_list = [], [], []

    while cap.isOpened():
        frame_batch = []
        for _ in range(batch_size):
            ret, frame = cap.read()
            if not ret:
                break
            frame_batch.append(frame)

        if not frame_batch:
            break

        # Fast NumPy conversion (BGR to RGB via slicing instead of cv2.cvtColor loop)
        block_arr = np.array(frame_batch, dtype=np.float32)
        block_arr = block_arr[:, :, :, ::-1]

        _, h, w, _ = block_arr.shape
        roi_block = block_arr[:, int(h*0.15):int(h*0.65), int(w*0.3):int(w*0.7), :]
        block_means = np.mean(roi_block, axis=(1, 2))

        red_list.extend(block_means[:, 0])
        green_list.extend(block_means[:, 1])
        blue_list.extend(block_means[:, 2])

    cap.release()
    if len(red_list) == 0:
        return None, None, None, 0

    return red_list, green_list, blue_list, fps

# --- 4. BUFFERED BATCH SAVING FOR MAXIMUM SPEED ---
success_count = 0
skipped_count = 0
buffer = []
BUFFER_FLUSH_SIZE = 1  # Write to disk every 50 videos to minimize I/O lag

if len(pending_videos) == 0:
    print("\n🎉 All available videos on disk have already been extracted and saved!")
else:
    with tqdm(pending_videos.items(), desc="High-Speed Extraction") as pbar:
        for filename, full_path in pbar:
            patient_id = "".join([char for char in filename if char.isdigit()])
            patient_id = int(patient_id[:4]) if len(patient_id) >= 4 else filename

            r, g, b, fps = extract_rppg_fast(full_path)

            if r is not None and len(r) > 0:
                buffer.append({
                    'patient_id': patient_id,
                    'video_file': filename,
                    'fps': fps,
                    'total_frames': len(r),
                    'r_signal': r,
                    'g_signal': g,
                    'b_signal': b
                })
                success_count += 1
            else:
                skipped_count += 1

            # Flush buffer to CSV in batches
            if len(buffer) >= BUFFER_FLUSH_SIZE:
                batch_df = pd.DataFrame(buffer)
                file_exists = os.path.exists(OUTPUT_RPPG_CSV)
                batch_df.to_csv(OUTPUT_RPPG_CSV, mode='a', index=False, header=not file_exists)
                buffer = []  # Clear buffer

            pbar.set_postfix(file=filename, saved=success_count, skipped=skipped_count)

    # Flush any remaining items in the buffer
    if len(buffer) > 0:
        batch_df = pd.DataFrame(buffer)
        file_exists = os.path.exists(OUTPUT_RPPG_CSV)
        batch_df.to_csv(OUTPUT_RPPG_CSV, mode='a', index=False, header=not file_exists)

print("\n==========================================")
print("HIGH-SPEED EXTRACTION COMPLETE!")
print(f" -> Newly Extracted & Saved: {success_count} videos")
print(f" -> Saved directly to:       {OUTPUT_RPPG_CSV}")
print("==========================================")

In [ ]:
import os
import pandas as pd

DATA_DIR = "/content/drive/MyDrive/Colab Notebooks/rPPG_Project/data"
OUTPUT_RPPG_CSV = os.path.join(DATA_DIR, "raw_rppg_signals.csv")

if os.path.exists(OUTPUT_RPPG_CSV):
    try:
        df = pd.read_csv(OUTPUT_RPPG_CSV, engine='python', on_bad_lines='skip')
        print(f"📊 Extraction Progress Report:")
        print(f" -> Total extracted video rows: {len(df)}")

        if 'patient_id' in df.columns:
            unique_patients = df['patient_id'].nunique()
            print(f" -> Total unique patients extracted: {unique_patients}")
        else:
            print("⚠️ 'patient_id' column not found in raw_rppg_signals.csv.")
    except Exception as e:
        print(f"Error reading CSV: {e}")
else:
    print("raw_rppg_signals.csv does not exist yet.")

In [ ]:
import os
import pandas as pd

# --- CONFIGURATION ---
DATA_DIR = "/content/drive/MyDrive/Colab Notebooks/rPPG_Project/data"
OUTPUT_RPPG_CSV = os.path.join(DATA_DIR, "raw_rppg_signals.csv")
VIDEO_EXTENSIONS = ('.mp4', '.avi', '.mov', '.mkv', '.webm')

# 1. Scan disk for all videos
disk_videos = set()
for root, dirs, files in os.walk(DATA_DIR):
    for file in files:
        if file.lower().endswith(VIDEO_EXTENSIONS):
            disk_videos.add(file)

# 2. Check what's already in the CSV
completed_videos = set()
if os.path.exists(OUTPUT_RPPG_CSV):
    try:
        df = pd.read_csv(OUTPUT_RPPG_CSV, engine='python', on_bad_lines='skip')
        if 'video_file' in df.columns:
            completed_videos = set(df['video_file'].dropna().unique())
        print(f"📊 Extraction Progress Report:")
        print(f" -> Total extracted video rows in CSV: {len(df)}")
        if 'patient_id' in df.columns:
            print(f" -> Unique patients extracted so far: {df['patient_id'].nunique()}")
    except Exception as e:
        print(f"⚠️ Error reading CSV: {e}")
else:
    print("⚠️ raw_rppg_signals.csv does not exist yet.")

# 3. Calculate remaining work
remaining_videos = disk_videos - completed_videos

print("-" * 40)
print(f" -> Total video files found on disk: {len(disk_videos)}")
print(f" -> Successfully saved in CSV:       {len(completed_videos)}")
print(f" -> Remaining videos left to extract: {len(remaining_videos)}")
print("-" * 40)

CHROM Filter

In [ ]:
import os
import numpy as np
import pandas as pd
from tqdm import tqdm

# --- 1. CONFIGURATION ---
DATA_DIR = "/content/drive/MyDrive/Colab Notebooks/rPPG_Project/data"
INPUT_CSV = os.path.join(DATA_DIR, "raw_rppg_signals.csv")
OUTPUT_CSV = os.path.join(DATA_DIR, "enhanced_rppg_signals.csv")

if not os.path.exists(INPUT_CSV):
    raise FileNotFoundError(f"Could not find raw signals file at: {INPUT_CSV}")

print("Loading raw signals CSV...")
raw_df = pd.read_csv(INPUT_CSV, engine='python', on_bad_lines='skip')
print(f" -> Total raw records found in CSV: {len(raw_df)}")

# --- 2. CHECK ALREADY ENHANCED VIDEOS ---
completed_enhanced = set()
if os.path.exists(OUTPUT_CSV):
    try:
        existing_enhanced_df = pd.read_csv(OUTPUT_CSV, engine='python', on_bad_lines='skip')
        if 'video_file' in existing_enhanced_df.columns:
            completed_enhanced = set(existing_enhanced_df['video_file'].dropna().unique())
        print(f" -> Found {len(completed_enhanced)} already enhanced videos in enhanced_rppg_signals.csv.")
    except Exception as e:
        print(f"⚠️ Warning: Could not fully parse existing enhanced CSV ({e}). Proceeding safely.")

pending_df = raw_df[~raw_df['video_file'].isin(completed_enhanced)].reset_index(drop=True)
print(f" -> Remaining videos to enhance: {len(pending_df)}")

# --- 3. HELPER FUNCTIONS (FIXED FOR np.float32) ---
def parse_signal(signal_input):
    """Safely convert stringified signal arrays back into numpy arrays."""
    if isinstance(signal_input, np.ndarray):
        return signal_input.astype(float)
    if isinstance(signal_input, list):
        return np.array(signal_input, dtype=float)
    if isinstance(signal_input, str):
        # Clean both np.float64 and np.float32 wrappers, brackets, and quotes
        cleaned = (signal_input.strip()
                   .replace("np.float64(", "").replace("np.float32(", "")
                   .replace(")", "").replace("[", "").replace("]", "")
                   .replace("'", "").replace('"', ""))
        if not cleaned:
            return np.array([])
        try:
            parts = [float(x.strip()) for x in cleaned.split(",") if x.strip()]
            if len(parts) > 0:
                return np.array(parts)
        except ValueError:
            pass
        try:
            parts = [float(x.strip()) for x in cleaned.split() if x.strip()]
            return np.array(parts)
        except ValueError:
            pass
    return np.array([], dtype=float)

def compute_chrom_rppg(r, g, b):
    """Applies Chrominance-based rPPG noise cancellation."""
    r_sig = parse_signal(r)
    g_sig = parse_signal(g)
    b_sig = parse_signal(b)

    if len(r_sig) == 0 or len(g_sig) == 0 or len(b_sig) == 0:
        return np.array([])

    xs = 3.0 * r_sig - 2.0 * g_sig
    ys = 1.5 * r_sig + g_sig - 1.5 * b_sig

    alpha = np.std(xs) / (np.std(ys) + 1e-6)
    chrom_signal = xs - alpha * ys
    return chrom_signal

# --- 4. SAFE INCREMENTAL PROCESSING ---
success_count = 0

if len(pending_df) == 0:
    print("\n🎉 All raw signals have already been enhanced with CHROM!")
else:
    print(f"Processing {len(pending_df)} remaining videos...")

    for idx, row in tqdm(pending_df.iterrows(), total=len(pending_df), desc="CHROM Enhancement"):
        chrom_sig = compute_chrom_rppg(row['r_signal'], row['g_signal'], row['b_signal'])

        if len(chrom_sig) > 0:
            single_record_df = pd.DataFrame([{
                'patient_id': row['patient_id'],
                'video_file': row['video_file'],
                'fps': row['fps'],
                'total_frames': row['total_frames'],
                'g_signal': row['g_signal'],
                'chrom_signal': chrom_sig.tolist()
            }])

            file_exists = os.path.exists(OUTPUT_CSV)
            single_record_df.to_csv(OUTPUT_CSV, mode='a', index=False, header=not file_exists)
            success_count += 1

print("\n==========================================")
print("INCREMENTAL CHROM ENHANCEMENT COMPLETE!")
print(f" -> Newly Processed & Saved: {success_count} videos")
print(f" -> Saved directly to:       {OUTPUT_CSV}")
print("==========================================")

Prediction model traning for Arterial Stiffness with LOSO

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error
import xgboost as xgb

# --- CONFIGURATION ---
DATA_DIR = "/content/drive/MyDrive/Colab Notebooks/rPPG_Project/data"
ENHANCED_CSV = os.path.join(DATA_DIR, "enhanced_rppg_signals.csv")
DB_CSV = os.path.join(DATA_DIR, "db.csv")
TARGET_PATIENT_LIMIT = 293

enhanced_df = pd.read_csv(ENHANCED_CSV)
db_df = pd.read_csv(DB_CSV)
FIXED_LENGTH = 900  # 30 seconds at 30 fps

# --- RESTRICT TO TARGET UNIQUE PATIENTS FROM ENHANCED CSV ---
if 'patient_id' in enhanced_df.columns:
    unique_patients = enhanced_df['patient_id'].dropna().unique()[:TARGET_PATIENT_LIMIT]
    enhanced_df = enhanced_df[enhanced_df['patient_id'].isin(unique_patients)].reset_index(drop=True)
    print(f" -> Restricted enhanced data to {len(unique_patients)} unique patients.")
else:
    enhanced_df = enhanced_df.head(TARGET_PATIENT_LIMIT)

# Safely encode categorical 'sex' column if present
if 'sex' in db_df.columns:
    db_df['sex_encoded'] = LabelEncoder().fit_transform(db_df['sex'].astype(str))
else:
    db_df['sex_encoded'] = 0

def parse_signal(signal_input):
    """Safely convert signal strings back to numpy arrays and pad/truncate to fixed length."""
    if isinstance(signal_input, np.ndarray):
        sig = signal_input.astype(float)
    elif isinstance(signal_input, list):
        sig = np.array(signal_input, dtype=float)
    elif isinstance(signal_input, str):
        cleaned = signal_input.strip().replace("np.float64(", "").replace(")", "").replace("[", "").replace("]", "").replace("'", "").replace('"', "")
        if not cleaned:
            return None
        try:
            parts = [float(x.strip()) for x in cleaned.split(",") if x.strip()]
            sig = np.array(parts, dtype=float)
        except ValueError:
            try:
                parts = [float(x.strip()) for x in cleaned.split() if x.strip()]
                sig = np.array(parts, dtype=float)
            except ValueError:
                return None
    else:
        return None

    if len(sig) == 0:
        return None

    if len(sig) < FIXED_LENGTH:
        sig = np.pad(sig, (0, FIXED_LENGTH - len(sig)), 'edge')
    else:
        sig = sig[:FIXED_LENGTH]
    return sig

# --- BUILD TENSORS & DEMOGRAPHICS ---
X_signals = []
demographics_list = []
y_rigidity = []

print("Preparing CHROM signals and demographics for Arterial Stiffness (Rigidity) 1D-CNN...")
for idx, row in enhanced_df.iterrows():
    video_file = row['video_file']
    chrom_sig = row['chrom_signal']

    match_row = db_df[db_df['video'].str.contains(video_file, na=False)]
    if match_row.empty or 'rigidity' not in match_row.columns:
        continue

    rigidity_val = match_row.iloc[0]['rigidity']
    if pd.isna(rigidity_val):
        continue

    age_val = match_row.iloc[0]['age'] if 'age' in match_row.columns and pd.notna(match_row.iloc[0]['age']) else 0.0
    sex_val = match_row.iloc[0]['sex_encoded'] if 'sex_encoded' in match_row.columns else 0.0

    processed_sig = parse_signal(chrom_sig)
    if processed_sig is not None:
        X_signals.append(processed_sig)
        demographics_list.append([float(age_val), float(sex_val)])
        y_rigidity.append(float(rigidity_val))

X_sig_arr = np.array(X_signals, dtype=np.float32)
X_demo_arr = np.array(demographics_list, dtype=np.float32)
y = np.array(y_rigidity, dtype=np.float32)

# Normalize signals and demographics
scaler_sig = StandardScaler()
X_sig_scaled = scaler_sig.fit_transform(X_sig_arr)

scaler_demo = StandardScaler()
X_demo_scaled = scaler_demo.fit_transform(X_demo_arr)

# Reshape signals for PyTorch Conv1D: [Batch, Channels=1, Length=900]
X_sig_scaled = np.expand_dims(X_sig_scaled, axis=1)

print(f"Dataset ready for CNN: {X_sig_scaled.shape[0]} valid samples across target pool.")

# --- HYBRID 1D-CNN ARCHITECTURE (WAVEFORM + DEMOGRAPHICS) ---
class Hybrid_RPPG_Rigidity_Net(nn.Module):
    def __init__(self):
        super(Hybrid_RPPG_Rigidity_Net, self).__init__()
        self.features = nn.Sequential(
            nn.Conv1d(1, 16, kernel_size=7, stride=2, padding=3),
            nn.BatchNorm1d(16),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(16, 32, kernel_size=5, stride=1, padding=2),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(32, 64, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1)  # Output shape: [Batch, 64, 1]
        )

        self.regressor = nn.Sequential(
            nn.Linear(64 + 2, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 1)
        )

    def forward(self, x_sig, x_demo):
        x_feat = self.features(x_sig)
        x_feat = x_feat.view(x_feat.size(0), -1)  # Flatten CNN features

        combined = torch.cat((x_feat, x_demo), dim=1)
        out = self.regressor(combined)
        return out.squeeze()

# --- K-FOLD CROSS VALIDATION FOR HYBRID CNN ---
kf = KFold(n_splits=5, shuffle=True, random_state=42)
fold_maes = []
fold_rmses = []
oof_preds = np.zeros_like(y, dtype=np.float32)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on device: {device}")

for fold, (train_idx, test_idx) in enumerate(kf.split(X_sig_scaled)):
    print(f"\n--- Training Fold {fold + 1}/5 ---")

    X_train_sig = torch.tensor(X_sig_scaled[train_idx], dtype=torch.float32)
    X_train_demo = torch.tensor(X_demo_scaled[train_idx], dtype=torch.float32)
    y_train = torch.tensor(y[train_idx], dtype=torch.float32)

    X_test_sig = torch.tensor(X_sig_scaled[test_idx], dtype=torch.float32)
    X_test_demo = torch.tensor(X_demo_scaled[test_idx], dtype=torch.float32)
    y_test = torch.tensor(y[test_idx], dtype=torch.float32)

    train_loader = DataLoader(TensorDataset(X_train_sig, X_train_demo, y_train), batch_size=16, shuffle=True)

    model = Hybrid_RPPG_Rigidity_Net().to(device)
    criterion = nn.L1Loss()  # MAE loss
    optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)

    # Training loop
    model.train()
    for epoch in range(25):
        for batch_sig, batch_demo, batch_y in train_loader:
            batch_sig, batch_demo, batch_y = batch_sig.to(device), batch_demo.to(device), batch_y.to(device)
            optimizer.zero_grad()
            outputs = model(batch_sig, batch_demo)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()

    # Evaluation
    model.eval()
    with torch.no_grad():
        test_sig_dev = X_test_sig.to(device)
        test_demo_dev = X_test_demo.to(device)
        test_y_dev = y_test.to(device)

        preds = model(test_sig_dev, test_demo_dev)

        oof_preds[test_idx] = preds.cpu().numpy()

        fold_mae = mean_absolute_error(test_y_dev.cpu().numpy(), preds.cpu().numpy())
        fold_rmse = np.sqrt(mean_squared_error(test_y_dev.cpu().numpy(), preds.cpu().numpy()))

        fold_maes.append(fold_mae)
        fold_rmses.append(fold_rmse)
        print(f"Fold {fold + 1} MAE: {fold_mae:.4f} | RMSE: {fold_rmse:.4f}")

print("\n==================================================")
print(f"HYBRID 1D-CNN ({TARGET_PATIENT_LIMIT} PATIENTS: CHROM + AGE + SEX) RESULTS:")
print("==================================================")
print(f" -> Mean MAE:  {np.mean(fold_maes):.4f} (+/- {np.std(fold_maes):.4f})")
print(f" -> Mean RMSE: {np.mean(fold_rmses):.4f} (+/- {np.std(fold_rmses):.4f})")

# --- 1. COMPUTE RESIDUALS ---
residuals = y - oof_preds

# --- 2. PREPARE FEATURES FOR THE SECONDARY MODEL ---
X_secondary = np.column_stack([oof_preds, X_demo_scaled])

# --- 3. TRAIN SECONDARY RESIDUAL MODEL (XGBoost) ---
print("\n--- Training Secondary Residual Correction Model (XGBoost) ---")
residual_model = xgb.XGBRegressor(
    n_estimators=100,
    learning_rate=0.05,
    max_depth=2,          # Keep shallow to prevent overfitting on errors
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

sec_kf = KFold(n_splits=5, shuffle=True, random_state=42)
sec_oof_residuals = np.zeros_like(residuals)

for train_idx, test_idx in sec_kf.split(X_secondary):
    X_sec_train, X_sec_test = X_secondary[train_idx], X_secondary[test_idx]
    y_res_train = residuals[train_idx]

    residual_model.fit(X_sec_train, y_res_train)
    sec_oof_residuals[test_idx] = residual_model.predict(X_sec_test)

# --- 4. COMPUTE FINAL HYBRID PREDICTIONS & EVALUATE ---
final_hybrid_preds = oof_preds + sec_oof_residuals

final_mae = mean_absolute_error(y, final_hybrid_preds)
final_rmse = np.sqrt(mean_squared_error(y, final_hybrid_preds))

print("\n==================================================")
print("FINAL TWO-STAGE HYBRID MODEL RESULTS (CNN + XGBoost Residuals):")
print("==================================================")
print(f" -> Baseline CNN Mean MAE:  {np.mean(fold_maes):.4f}")
print(f" -> Final Two-Stage MAE:    {final_mae:.4f}")
print(f" -> Final Two-Stage RMSE:   {final_rmse:.4f}")

Scatter plot for comparison


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# --- 1. CALCULATE ERRORS & IDENTIFY OUTLIERS ---
errors = np.abs(y - final_hybrid_preds)

# Define a threshold for outliers (e.g., top 5% highest error or error > 2 * standard deviation)
threshold = np.mean(errors) + 2 * np.std(errors)
outlier_indices = np.where(errors > threshold)[0]

print(f"📊 Outlier Analysis:")
print(f" -> Mean Absolute Error: {np.mean(errors):.4f}")
print(f" -> Outlier Threshold (> 2 std dev): {threshold:.4f}")
print(f" -> Number of flagged outliers: {len(outlier_indices)}")

# --- 2. GENERATE HIGHLIGHTED SCATTER PLOT ---
sns.set_theme(style="whitegrid")
plt.figure(figsize=(9, 9))

# Separate normal predictions and outliers for color-coding
normal_mask = errors <= threshold
outlier_mask = errors > threshold

# Plot normal predictions
plt.scatter(y[normal_mask], final_hybrid_preds[normal_mask],
            alpha=0.5, color='dodgerblue', edgecolor='k', s=45, label='Normal Predictions')

# Plot heavily mispredicted outliers in contrasting crimson/orange
plt.scatter(y[outlier_mask], final_hybrid_preds[outlier_mask],
            alpha=0.9, color='crimson', edgecolor='k', s=80, marker='X', label=f'Outliers (Error > {threshold:.2f})')

# Plot ideal 45-degree reference line (y = x)
min_val = min(min(y), min(final_hybrid_preds))
max_val = max(max(y), max(final_hybrid_preds))
plt.plot([min_val, max_val], [min_val, max_val], color='black', linestyle='--', linewidth=2, label='Ideal Fit (y = x)')

# Labels and formatting
plt.title('Actual vs. Predicted Arterial Stiffness (Highlighted Outliers)', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Actual Rigidity (from db.csv)', fontsize=12)
plt.ylabel('Predicted Rigidity (Hybrid Model)', fontsize=12)
plt.legend(loc='upper left', frameon=True, fontsize=11)

# Annotate performance metrics
metrics_text = f"Final MAE: {final_mae:.4f}\nFinal RMSE: {final_rmse:.4f}\nOutliers Flagged: {len(outlier_indices)}"
plt.gca().text(0.05, 0.80, metrics_text, transform=plt.gca().transAxes,
               fontsize=11, bbox=dict(boxstyle='round,pad=0.5', facecolor='white', alpha=0.9, edgecolor='gray'))

plt.tight_layout()
plt.show()

# --- 3. PRINT SUMMARY TABLE OF THE WORST PREDICTIONS ---
print("\n==================================================")
print("TOP WORST PREDICTIONS (HIGHEST ERRORS):")
print("==================================================")
worst_sorted_idx = np.argsort(errors)[::-1][:10] # Top 10 worst cases

summary_data = []
for idx in worst_sorted_idx:
    summary_data.append({
        'Sample Index': idx,
        'Actual': y[idx],
        'Predicted': final_hybrid_preds[idx],
        'Absolute Error': errors[idx]
    })

summary_df = pd.DataFrame(summary_data)
print(summary_df.to_string(index=False))

BMI with CHROM, LOSO and 1DCNN


In [ ]:
import os
import ast
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error
import xgboost as xgb

# --- 1. CONFIGURATION ---
DATA_DIR = "/content/drive/MyDrive/Colab Notebooks/rPPG_Project/data"
ENHANCED_CSV = os.path.join(DATA_DIR, "enhanced_rppg_signals.csv")
DB_CSV = os.path.join(DATA_DIR, "db.csv")
TARGET_PATIENT_LIMIT = 293
FIXED_LENGTH = 900  # 30 seconds at 30 fps

print("Loading datasets...")
enhanced_df = pd.read_csv(ENHANCED_CSV)
db_df = pd.read_csv(DB_CSV)

# --- 2. RESTRICT TO TARGET UNIQUE PATIENTS ---
if 'patient_id' in enhanced_df.columns:
    unique_patients = enhanced_df['patient_id'].dropna().unique()[:TARGET_PATIENT_LIMIT]
    enhanced_df = enhanced_df[enhanced_df['patient_id'].isin(unique_patients)].reset_index(drop=True)
    print(f" -> Restricted enhanced data to {len(unique_patients)} unique patients ({len(enhanced_df)} total video rows).")
else:
    enhanced_df = enhanced_df.head(TARGET_PATIENT_LIMIT)

# Safely encode categorical 'sex' column if present
if 'sex' in db_df.columns:
    db_df['sex_encoded'] = LabelEncoder().fit_transform(db_df['sex'].astype(str))
else:
    db_df['sex_encoded'] = 0

def parse_signal(signal_input):
    """Safely convert signal strings back to numpy arrays and pad/truncate."""
    if isinstance(signal_input, np.ndarray):
        sig = signal_input.astype(float)
    elif isinstance(signal_input, list):
        sig = np.array(signal_input, dtype=float)
    elif isinstance(signal_input, str):
        cleaned = (signal_input.strip()
                   .replace("np.float64(", "").replace("np.float32(", "")
                   .replace(")", "").replace("[", "").replace("]", "")
                   .replace("'", "").replace('"', ""))
        if not cleaned:
            return None
        try:
            parts = [float(x.strip()) for x in cleaned.split(",") if x.strip()]
            sig = np.array(parts, dtype=float)
        except ValueError:
            try:
                parts = [float(x.strip()) for x in cleaned.split() if x.strip()]
                sig = np.array(parts, dtype=float)
            except ValueError:
                return None
    else:
        return None

    if len(sig) == 0:
        return None

    if len(sig) < FIXED_LENGTH:
        sig = np.pad(sig, (0, FIXED_LENGTH - len(sig)), 'edge')
    else:
        sig = sig[:FIXED_LENGTH]
    return sig

# --- 3. BUILD TENSORS, DEMOGRAPHICS & GROUPS ---
X_signals = []
demographics_list = []
y_target = []
patient_groups = []

print("Preparing CHROM signals, demographics, and patient grouping...")
for idx, row in enhanced_df.iterrows():
    video_file = row['video_file']
    chrom_sig = row['chrom_signal']
    patient_id = row['patient_id']

    match_row = db_df[db_df['video'].str.contains(str(video_file), regex=False, na=False)]
    if match_row.empty or 'rigidity' not in match_row.columns:
        continue

    rigidity_val = match_row.iloc[0]['rigidity']
    if pd.isna(rigidity_val):
        continue

    age_val = match_row.iloc[0]['age'] if 'age' in match_row.columns and pd.notna(match_row.iloc[0]['age']) else 0.0
    sex_val = match_row.iloc[0]['sex_encoded'] if 'sex_encoded' in match_row.columns else 0.0

    processed_sig = parse_signal(chrom_sig)
    if processed_sig is not None:
        X_signals.append(processed_sig)
        demographics_list.append([float(age_val), float(sex_val)])
        y_target.append(float(rigidity_val))
        patient_groups.append(patient_id)

X_sig_arr = np.array(X_signals, dtype=np.float32)
X_demo_arr = np.array(demographics_list, dtype=np.float32)
y = np.array(y_target, dtype=np.float32)
groups = np.array(patient_groups)

# Normalize signals and demographics
scaler_sig = StandardScaler()
X_sig_scaled = scaler_sig.fit_transform(X_sig_arr)
X_sig_scaled = np.expand_dims(X_sig_scaled, axis=1)

scaler_demo = StandardScaler()
X_demo_scaled = scaler_demo.fit_transform(X_demo_arr)

print(f"Dataset compiled: {X_sig_scaled.shape[0]} valid samples across {len(np.unique(groups))} unique patients.")

# --- 4. HYBRID 1D-CNN ARCHITECTURE ---
class Hybrid_RPPG_Rigidity_Net(nn.Module):
    def __init__(self):
        super(Hybrid_RPPG_Rigidity_Net, self).__init__()
        self.features = nn.Sequential(
            nn.Conv1d(1, 16, kernel_size=7, stride=2, padding=3),
            nn.BatchNorm1d(16),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(16, 32, kernel_size=5, stride=1, padding=2),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(32, 64, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1)
        )

        self.regressor = nn.Sequential(
            nn.Linear(64 + 2, 32),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(32, 1)
        )

    def forward(self, x_sig, x_demo):
        x_feat = self.features(x_sig)
        x_feat = x_feat.view(x_feat.size(0), -1)
        combined = torch.cat((x_feat, x_demo), dim=1)
        out = self.regressor(combined)
        return out.squeeze()

# --- 5. GROUP 5-FOLD CROSS VALIDATION (CNN) ---
gkf = GroupKFold(n_splits=5)
fold_maes = []
fold_rmses = []
oof_preds = np.zeros_like(y, dtype=np.float32)  # Initialized to store OOF predictions

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on device: {device}")

for fold, (train_idx, test_idx) in enumerate(gkf.split(X_sig_scaled, y, groups=groups)):
    print(f"\n--- Training Fold {fold + 1}/5 (Patient-Independent Split) ---")

    X_train_sig = torch.tensor(X_sig_scaled[train_idx], dtype=torch.float32)
    X_train_demo = torch.tensor(X_demo_scaled[train_idx], dtype=torch.float32)
    y_train = torch.tensor(y[train_idx], dtype=torch.float32)

    X_test_sig = torch.tensor(X_sig_scaled[test_idx], dtype=torch.float32)
    X_test_demo = torch.tensor(X_demo_scaled[test_idx], dtype=torch.float32)
    y_test = torch.tensor(y[test_idx], dtype=torch.float32)

    train_loader = DataLoader(TensorDataset(X_train_sig, X_train_demo, y_train), batch_size=16, shuffle=True)

    model = Hybrid_RPPG_Rigidity_Net().to(device)
    criterion = nn.L1Loss()
    optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)

    # Training loop
    model.train()
    for epoch in range(30):
        for batch_sig, batch_demo, batch_y in train_loader:
            batch_sig, batch_demo, batch_y = batch_sig.to(device), batch_demo.to(device), batch_y.to(device)
            optimizer.zero_grad()
            outputs = model(batch_sig, batch_demo)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()

    # Evaluation
    model.eval()
    with torch.no_grad():
        test_sig_dev = X_test_sig.to(device)
        test_demo_dev = X_test_demo.to(device)
        test_y_dev = y_test.to(device)

        preds = model(test_sig_dev, test_demo_dev)

        # Save Out-Of-Fold predictions
        oof_preds[test_idx] = preds.cpu().numpy()

        fold_mae = mean_absolute_error(test_y_dev.cpu().numpy(), preds.cpu().numpy())
        fold_rmse = np.sqrt(mean_squared_error(test_y_dev.cpu().numpy(), preds.cpu().numpy()))

        fold_maes.append(fold_mae)
        fold_rmses.append(fold_rmse)
        print(f" -> Fold {fold + 1} MAE: {fold_mae:.4f} | RMSE: {fold_rmse:.4f}")

print("\n==================================================")
print(f"HYBRID 1D-CNN ({TARGET_PATIENT_LIMIT} PATIENTS GROUPKFOLD) RESULTS:")
print("==================================================")
print(f" -> Mean MAE:  {np.mean(fold_maes):.4f} (+/- {np.std(fold_maes):.4f})")
print(f" -> Mean RMSE: {np.mean(fold_rmses):.4f} (+/- {np.std(fold_rmses):.4f})")

# --- 6. SECONDARY RESIDUAL CORRECTION (XGBOOST) ---
print("\n--- Training Secondary Residual Correction Model (XGBoost) ---")
residuals = y - oof_preds
X_secondary = np.column_stack([oof_preds, X_demo_scaled])

residual_model = xgb.XGBRegressor(
    n_estimators=100,
    learning_rate=0.05,
    max_depth=3,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

sec_gkf = GroupKFold(n_splits=5)
sec_oof_residuals = np.zeros_like(residuals)

for train_idx, test_idx in sec_gkf.split(X_secondary, residuals, groups=groups):
    X_sec_train, X_sec_test = X_secondary[train_idx], X_secondary[test_idx]
    y_res_train = residuals[train_idx]

    residual_model.fit(X_sec_train, y_res_train)
    sec_oof_residuals[test_idx] = residual_model.predict(X_sec_test)

final_hybrid_preds = oof_preds + sec_oof_residuals
final_mae = mean_absolute_error(y, final_hybrid_preds)
final_rmse = np.sqrt(mean_squared_error(y, final_hybrid_preds))

print("\n==================================================")
print("FINAL TWO-STAGE HYBRID MODEL RESULTS (CNN + XGBoost Residuals):")
print("==================================================")
print(f" -> Baseline CNN Mean MAE:  {np.mean(fold_maes):.4f}")
print(f" -> Final Two-Stage MAE:    {final_mae:.4f}")
print(f" -> Final Two-Stage RMSE:   {final_rmse:.4f}")
mae_improvement = ((np.mean(fold_maes) - final_mae) / np.mean(fold_maes)) * 100
print(f" Net MAE Improvement: {mae_improvement:+.2f}%")
print("==================================================")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# --- CALCULATE ERRORS & IDENTIFY OUTLIERS ---
errors = np.abs(y - final_hybrid_preds)
threshold = np.mean(errors) + 2 * np.std(errors)
outlier_mask = errors > threshold
normal_mask = errors <= threshold

print(f"📊 BMI Outlier Analysis:")
print(f" -> Outlier Threshold (> 2 std dev): {threshold:.4f}")
print(f" -> Number of flagged outliers: {outlier_mask.sum()}")

# --- GENERATE HIGHLIGHTED SCATTER PLOT ---
sns.set_theme(style="whitegrid")
plt.figure(figsize=(9, 9))

# Plot normal predictions
plt.scatter(y[normal_mask], final_hybrid_preds[normal_mask],
            alpha=0.5, color='dodgerblue', edgecolor='k', s=45, label='Normal Predictions')

# Plot outliers in crimson with an 'X' marker
plt.scatter(y[outlier_mask], final_hybrid_preds[outlier_mask],
            alpha=0.9, color='crimson', edgecolor='k', s=80, marker='X', label=f'Outliers (Error > {threshold:.2f})')

# Plot ideal 45-degree reference line (y = x)
min_val = min(y.min(), final_hybrid_preds.min())
max_val = max(y.max(), final_hybrid_preds.max())
plt.plot([min_val, max_val], [min_val, max_val], color='black', linestyle='--', linewidth=2, label='Ideal Fit (y = x)')

# Labels and formatting
plt.title('Actual vs. Predicted BMI (Highlighted Outliers)', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Actual BMI (from db.csv)', fontsize=12)
plt.ylabel('Predicted BMI (Hybrid Model)', fontsize=12)
plt.legend(loc='upper left', frameon=True, fontsize=11)

# Annotate performance metrics
metrics_text = f"Final MAE: {final_mae:.4f}\nFinal RMSE: {final_rmse:.4f}\nOutliers Flagged: {outlier_mask.sum()}"
plt.gca().text(0.05, 0.80, metrics_text, transform=plt.gca().transAxes,
               fontsize=11, bbox=dict(boxstyle='round,pad=0.5', facecolor='white', alpha=0.9, edgecolor='gray'))

plt.tight_layout()
plt.show()